# Huấn luyện DeepVQE D3 GRU768 trên Kaggle — Phase 2

Notebook này train/eval D3_hybrid_head trên cùng Kaggle protocol: dùng Kaggle Working Dir, dùng Local SSD `/kaggle/working/data` để train nhanh, và lưu checkpoint/kết quả trong Kaggle Working Dir.

Notebook chỉ train và đánh giá riêng `D3_hybrid_head = Baseline architecture + bottleneck GRU hidden 768`. D3 giữ nguyên encoder/decoder/residual/CCM và chỉ tăng capacity ở bottleneck temporal context.


## 1. Cài đặt môi trường & chuẩn bị Kaggle Working Dir


In [ ]:
!pip install -q wandb gdown matplotlib tensorboard soundfile pandas tqdm einops pesq pystoi pyyaml torchmetrics


import os
import sys
from pathlib import Path

# Nếu bạn đang dùng workspace khác, đổi dòng này trước khi chạy tiếp.
WORK_DIR = Path('/kaggle/working/DeepVQE_Workspace')
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)

SHARED_CHECKPOINTS_DIR = Path('/kaggle/working/DeepVQE_Workspace/checkpoints')
SHARED_CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)

print(f'Thư mục làm việc hiện tại: {Path.cwd()}')
print(f'Python executable: {sys.executable}')
print(f'Checkpoint root: {SHARED_CHECKPOINTS_DIR}')


## 2. Clone mã nguồn DeepVQE


In [ ]:
# Repo này cần có deepvqe.py, ablation/ và stream/.
import subprocess
import importlib

GIT_REPO = 'https://github.com/hoxuanphu/Pdeepvqe.git'
REPO_DIR = WORK_DIR / 'deepvqe'

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', GIT_REPO, str(REPO_DIR)], check=True)
else:
    print(f'Thư mục {REPO_DIR} đã tồn tại. Đang cập nhật code mới nhất...')
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=False)

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

# Repo clone có thể chưa có D3_hybrid_head. Patch trực tiếp trong runtime trước mọi import ablation.
patch_path = Path('ablation/deepvqe_ablation.py')
patch_path.parent.mkdir(parents=True, exist_ok=True)
patch_path.write_text('"""Parameterized DeepVQE ablation variants.\n\nThis module intentionally leaves ``deepvqe.py`` untouched. The Baseline\nconfiguration is state-dict compatible with ``deepvqe.DeepVQE`` and can load\nbaseline weights with ``strict=True``. Non-baseline variants change module\nstructure and should load baseline weights with ``strict=False`` only.\n"""\n\nfrom copy import deepcopy\n\nimport torch\nimport torch.nn as nn\nfrom einops import rearrange\n\nfrom deepvqe import CCM, FE, SubpixelConv2d\nfrom stream.modules.convolution import StreamConv2d\n\n\nBASE_GRU_HIDDEN = 64 * 9\n\n\nABLATION_CONFIGS = {\n    "Baseline": {\n        "prelu_type": None,\n        "dw_residual": False,\n        "use_eca_f": False,\n        "main_block_eca_f": False,\n        "gru_hidden": BASE_GRU_HIDDEN,\n    },\n    "D1a_gru704": {\n        "prelu_type": None,\n        "dw_residual": False,\n        "use_eca_f": False,\n        "main_block_eca_f": False,\n        "skip_gate": None,\n        "dw_subpixel": False,\n        "gru_hidden": 704,\n    },\n    "D3_hybrid_head": {\n        "prelu_type": None,\n        "dw_residual": False,\n        "use_eca_f": False,\n        "main_block_eca_f": False,\n        "skip_gate": None,\n        "dw_subpixel": False,\n        "gru_hidden": 768,\n    },\n    "B1a": {\n        "prelu_type": "shared",\n        "dw_residual": False,\n        "use_eca_f": False,\n        "main_block_eca_f": False,\n        "gru_hidden": BASE_GRU_HIDDEN,\n    },\n    "B1b": {\n        "prelu_type": "per_channel",\n        "dw_residual": False,\n        "use_eca_f": False,\n        "main_block_eca_f": False,\n        "gru_hidden": BASE_GRU_HIDDEN,\n    },\n    "B2": {\n        "prelu_type": None,\n        "dw_residual": False,\n        "use_eca_f": True,\n        "main_block_eca_f": False,\n        "skip_gate": None,\n        "dw_subpixel": False,\n        "gru_hidden": BASE_GRU_HIDDEN,\n    },\n    "B3a": {\n        "prelu_type": None,\n        "dw_residual": False,\n        "use_eca_f": False,\n        "main_block_eca_f": False,\n        "skip_gate": "eca_f",\n        "dw_subpixel": False,\n        "gru_hidden": BASE_GRU_HIDDEN,\n    },\n    "B3b": {\n        "prelu_type": None,\n        "dw_residual": False,\n        "use_eca_f": False,\n        "main_block_eca_f": False,\n        "skip_gate": "se_f",\n        "dw_subpixel": False,\n        "gru_hidden": BASE_GRU_HIDDEN,\n    },\n    "C1": {\n        "prelu_type": None,\n        "dw_residual": True,\n        "use_eca_f": False,\n        "main_block_eca_f": False,\n        "skip_gate": None,\n        "dw_subpixel": False,\n        "gru_hidden": BASE_GRU_HIDDEN,\n    },\n    "C2a": {\n        "prelu_type": None,\n        "dw_residual": True,\n        "use_eca_f": False,\n        "main_block_eca_f": False,\n        "skip_gate": None,\n        "dw_subpixel": True,\n        "gru_hidden": BASE_GRU_HIDDEN,\n    },\n    "C2b": {\n        "prelu_type": None,\n        "dw_residual": True,\n        "use_eca_f": True,\n        "main_block_eca_f": False,\n        "skip_gate": None,\n        "dw_subpixel": False,\n        "gru_hidden": BASE_GRU_HIDDEN,\n    },\n    # C3/C4 depend on the B1 tie-breaker. Per-channel matches the previous\n    # local default and can be overridden to "shared" once B1a wins.\n    "C3": {\n        "prelu_type": "per_channel",\n        "dw_residual": True,\n        "use_eca_f": False,\n        "main_block_eca_f": False,\n        "skip_gate": None,\n        "dw_subpixel": False,\n        "gru_hidden": BASE_GRU_HIDDEN,\n    },\n    "C4": {\n        "prelu_type": "per_channel",\n        "dw_residual": True,\n        "use_eca_f": True,\n        "main_block_eca_f": False,\n        "skip_gate": None,\n        "dw_subpixel": False,\n        "gru_hidden": BASE_GRU_HIDDEN,\n    },\n}\n\n\nLEGACY_CONFIG_ALIASES = {\n    "C1b": "C1",\n    "C2": "C2b",\n}\n\n\nLEGACY_CONFIGS = {\n    "C1a-g2": {\n        "prelu_type": None,\n        "dw_residual": False,\n        "res_groups": 2,\n        "use_eca_f": False,\n        "main_block_eca_f": False,\n        "skip_gate": None,\n        "dw_subpixel": False,\n        "gru_hidden": BASE_GRU_HIDDEN,\n    },\n    "C1a-g4": {\n        "prelu_type": None,\n        "dw_residual": False,\n        "res_groups": 4,\n        "use_eca_f": False,\n        "main_block_eca_f": False,\n        "skip_gate": None,\n        "dw_subpixel": False,\n        "gru_hidden": BASE_GRU_HIDDEN,\n    },\n    # Legacy B3 was ECA-F in main encoder/decoder blocks. Roadmap B3a/B3b\n    # now cover skip gating, but this remains loadable for older runs.\n    "B3": {\n        "prelu_type": None,\n        "dw_residual": False,\n        "use_eca_f": True,\n        "main_block_eca_f": True,\n        "skip_gate": None,\n        "dw_subpixel": False,\n        "gru_hidden": BASE_GRU_HIDDEN,\n    },\n}\n\n\ndef _activation_to_prelu_type(activation):\n    if activation in (None, "elu"):\n        return None\n    if activation == "prelu_shared":\n        return "shared"\n    if activation == "prelu_channel":\n        return "per_channel"\n    if activation in ("shared", "per_channel"):\n        return activation\n    raise ValueError(\n        f"Unsupported activation {activation!r}; expected \'elu\', "\n        "\'prelu_shared\', or \'prelu_channel\'."\n    )\n\n\ndef _normalize_model_config(config):\n    """Accept planned config keys plus legacy local keys."""\n    cfg = deepcopy(config)\n\n    if "activation" in cfg:\n        activation = cfg.pop("activation")\n        legacy_prelu_type = _activation_to_prelu_type(activation)\n        if legacy_prelu_type is not None or cfg.get("prelu_type") is None:\n            cfg["prelu_type"] = legacy_prelu_type\n\n    if "attn_type" in cfg:\n        attn_type = cfg.pop("attn_type")\n        if attn_type in (None, "none"):\n            cfg["use_eca_f"] = bool(cfg.get("use_eca_f", False))\n        elif attn_type == "eca_f":\n            cfg["use_eca_f"] = True\n        else:\n            raise ValueError(\n                "Only attn_type=\'eca_f\' is supported in Phase 1 because "\n                "ECA-CT has no streaming cache contract."\n            )\n\n    if "res_conv_type" in cfg:\n        res_conv_type = cfg.pop("res_conv_type")\n        if res_conv_type == "standard":\n            cfg["dw_residual"] = bool(cfg.get("dw_residual", False))\n        elif res_conv_type == "dw_separable":\n            cfg["dw_residual"] = True\n        elif res_conv_type == "grouped":\n            cfg["dw_residual"] = False\n        else:\n            raise ValueError(\n                f"Unsupported res_conv_type {res_conv_type!r}; expected "\n                "\'standard\', \'grouped\', or \'dw_separable\'."\n            )\n\n    cfg.setdefault("prelu_type", None)\n    cfg.setdefault("dw_residual", False)\n    cfg.setdefault("use_eca_f", False)\n    cfg.setdefault("main_block_eca_f", False)\n    cfg.setdefault("gru_hidden", BASE_GRU_HIDDEN)\n    cfg.setdefault("res_groups", None)\n    cfg.setdefault("skip_gate", None)\n    cfg.setdefault("dw_subpixel", False)\n\n    if cfg["skip_gate"] in ("none", "identity", False):\n        cfg["skip_gate"] = None\n    if cfg["skip_gate"] == "se":\n        cfg["skip_gate"] = "se_f"\n    if cfg["skip_gate"] not in (None, "eca_f", "se_f"):\n        raise ValueError("skip_gate must be None, \'eca_f\', or \'se_f\'")\n\n    allowed = {\n        "prelu_type",\n        "dw_residual",\n        "use_eca_f",\n        "main_block_eca_f",\n        "gru_hidden",\n        "res_groups",\n        "skip_gate",\n        "dw_subpixel",\n    }\n    unknown = sorted(set(cfg) - allowed)\n    if unknown:\n        raise ValueError(f"Unknown model config keys: {unknown}")\n    return cfg\n\n\ndef get_ablation_config(config_id="Baseline", **overrides):\n    """Return a copy of a named ablation config with optional overrides."""\n    if config_id in LEGACY_CONFIG_ALIASES:\n        config_id = LEGACY_CONFIG_ALIASES[config_id]\n\n    if config_id in ABLATION_CONFIGS:\n        config = deepcopy(ABLATION_CONFIGS[config_id])\n    elif config_id in LEGACY_CONFIGS:\n        config = deepcopy(LEGACY_CONFIGS[config_id])\n    else:\n        valid = ", ".join(list(ABLATION_CONFIGS) + list(LEGACY_CONFIG_ALIASES) + list(LEGACY_CONFIGS))\n        raise ValueError(f"Unknown ablation config {config_id!r}. Valid configs: {valid}")\n\n    config.update(overrides)\n    return _normalize_model_config(config)\n\n\ndef ActivationFactory(prelu_type=None, channels=None):\n    """Build the configured activation for ``(B, C, T, F)`` feature tensors."""\n    prelu_type = _activation_to_prelu_type(prelu_type)\n    if prelu_type is None:\n        return nn.ELU()\n    if prelu_type == "shared":\n        return nn.PReLU(num_parameters=1)\n    if prelu_type == "per_channel":\n        if channels is None:\n            raise ValueError("channels must be provided for prelu_type=\'per_channel\'")\n        return nn.PReLU(num_parameters=int(channels))\n    raise ValueError("prelu_type must be None, \'shared\', or \'per_channel\'")\n\n\nclass CausalECA_F(nn.Module):\n    """Frequency-pooled efficient channel attention.\n\n    Pooling is only over the frequency axis. The 1-D convolution slides across\n    channels for each frame independently, so no temporal cache is required.\n    """\n\n    def __init__(self, channels, kernel_size=3):\n        super().__init__()\n        if kernel_size % 2 == 0:\n            raise ValueError("CausalECA_F kernel_size must be odd")\n        self.channels = int(channels)\n        self.conv = nn.Conv1d(\n            1,\n            1,\n            kernel_size=kernel_size,\n            padding=(kernel_size - 1) // 2,\n            bias=False,\n        )\n        self.sigmoid = nn.Sigmoid()\n\n    def forward(self, x):\n        """x: (B, C, T, F)."""\n        b, c, t, _ = x.shape\n        if c != self.channels:\n            raise ValueError(f"CausalECA_F expected {self.channels} channels, got {c}")\n\n        y = x.mean(dim=3, keepdim=True)  # (B, C, T, 1)\n        y = y.squeeze(3).permute(0, 2, 1).reshape(b * t, 1, c)\n        y = self.sigmoid(self.conv(y))\n        y = y.reshape(b, t, c).permute(0, 2, 1).unsqueeze(3)\n        return x * y\n\n\nECA_F = CausalECA_F\n\n\ndef _attention(channels, enabled):\n    return CausalECA_F(channels) if enabled else nn.Identity()\n\n\nclass FrequencySE(nn.Module):\n    """Frequency-only squeeze/excitation for causal skip gating."""\n\n    def __init__(self, channels, reduction=8):\n        super().__init__()\n        hidden = max(1, int(channels) // int(reduction))\n        self.gate = nn.Sequential(\n            nn.Conv2d(channels, hidden, kernel_size=1),\n            nn.ReLU(inplace=True),\n            nn.Conv2d(hidden, channels, kernel_size=1),\n            nn.Sigmoid(),\n        )\n\n    def forward(self, x):\n        y = x.mean(dim=3, keepdim=True)\n        return x * self.gate(y)\n\n\ndef _skip_gate(channels, gate_type):\n    if gate_type is None:\n        return nn.Identity()\n    if gate_type == "eca_f":\n        return CausalECA_F(channels)\n    if gate_type == "se_f":\n        return FrequencySE(channels)\n    raise ValueError(f"Unsupported skip_gate={gate_type!r}")\n\n\nclass DWSubpixelConv2d(nn.Module):\n    """Depthwise-separable version of the original SubpixelConv2d."""\n\n    def __init__(self, in_channels, out_channels, kernel_size=(4, 3)):\n        super().__init__()\n        self.pad = nn.ZeroPad2d([1, 1, 3, 0])\n        self.depthwise = nn.Conv2d(in_channels, in_channels, kernel_size, groups=in_channels)\n        self.pointwise = nn.Conv2d(in_channels, out_channels * 2, kernel_size=1)\n\n    def forward(self, x):\n        y = self.pointwise(self.depthwise(self.pad(x)))\n        y = rearrange(y, "b (r c) t f -> b c t (r f)", r=2)\n        return y\n\n\nclass ResidualBlock_Ablation(nn.Module):\n    def __init__(\n        self,\n        channels,\n        prelu_type=None,\n        dw_residual=False,\n        use_eca_f=False,\n        res_groups=None,\n    ):\n        super().__init__()\n        channels = int(channels)\n        self.pad = nn.ZeroPad2d([1, 1, 3, 0])\n        self.dw_residual = bool(dw_residual)\n        self.res_groups = res_groups\n\n        if self.dw_residual:\n            self.depthwise = nn.Conv2d(channels, channels, kernel_size=(4, 3), groups=channels)\n            self.pointwise = nn.Conv2d(channels, channels, kernel_size=1)\n        else:\n            groups = int(res_groups) if res_groups is not None else 1\n            if channels % groups != 0:\n                raise ValueError(f"channels={channels} is not divisible by res_groups={groups}")\n            self.conv = nn.Conv2d(channels, channels, kernel_size=(4, 3), groups=groups)\n\n        self.bn = nn.BatchNorm2d(channels)\n        self.elu = ActivationFactory(prelu_type, channels)\n        self.eca_f = _attention(channels, use_eca_f)\n\n    def forward(self, x):\n        """x: (B, C, T, F)."""\n        if self.dw_residual:\n            y = self.pointwise(self.depthwise(self.pad(x)))\n        else:\n            y = self.conv(self.pad(x))\n        y = self.eca_f(self.elu(self.bn(y)))\n        return y + x\n\n\nclass EncoderBlock_Ablation(nn.Module):\n    def __init__(\n        self,\n        in_channels,\n        out_channels,\n        kernel_size=(4, 3),\n        stride=(1, 2),\n        prelu_type=None,\n        dw_residual=False,\n        use_eca_f=False,\n        main_block_eca_f=False,\n        res_groups=None,\n    ):\n        super().__init__()\n        self.pad = nn.ZeroPad2d([1, 1, 3, 0])\n        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size, stride=stride)\n        self.bn = nn.BatchNorm2d(out_channels)\n        self.elu = ActivationFactory(prelu_type, out_channels)\n        self.main_eca_f = _attention(out_channels, main_block_eca_f)\n        self.resblock = ResidualBlock_Ablation(\n            out_channels,\n            prelu_type=prelu_type,\n            dw_residual=dw_residual,\n            use_eca_f=use_eca_f,\n            res_groups=res_groups,\n        )\n\n    def forward(self, x):\n        y = self.elu(self.bn(self.conv(self.pad(x))))\n        y = self.main_eca_f(y)\n        return self.resblock(y)\n\n\nclass Bottleneck_Ablation(nn.Module):\n    def __init__(self, input_size, hidden_size):\n        super().__init__()\n        self.gru = nn.GRU(input_size, hidden_size, batch_first=True)\n        self.fc = nn.Linear(hidden_size, input_size)\n\n    def forward(self, x):\n        """x: (B, C, T, F)."""\n        y = rearrange(x, "b c t f -> b t (c f)")\n        y = self.gru(y)[0]\n        y = self.fc(y)\n        y = rearrange(y, "b t (c f) -> b c t f", c=x.shape[1])\n        return y\n\n\nclass DecoderBlock_Ablation(nn.Module):\n    def __init__(\n        self,\n        in_channels,\n        out_channels,\n        kernel_size=(4, 3),\n        is_last=False,\n        prelu_type=None,\n        dw_residual=False,\n        use_eca_f=False,\n        main_block_eca_f=False,\n        res_groups=None,\n        skip_gate=None,\n        dw_subpixel=False,\n    ):\n        super().__init__()\n        self.skip_conv = nn.Conv2d(in_channels, in_channels, 1)\n        self.skip_gate = _skip_gate(in_channels, skip_gate)\n        self.resblock = ResidualBlock_Ablation(\n            in_channels,\n            prelu_type=prelu_type,\n            dw_residual=dw_residual,\n            use_eca_f=use_eca_f,\n            res_groups=res_groups,\n        )\n        if dw_subpixel:\n            self.deconv = DWSubpixelConv2d(in_channels, out_channels, kernel_size)\n        else:\n            self.deconv = SubpixelConv2d(in_channels, out_channels, kernel_size)\n        self.bn = nn.BatchNorm2d(out_channels)\n        self.elu = ActivationFactory(prelu_type, out_channels)\n        self.is_last = is_last\n        self.main_eca_f = _attention(out_channels, main_block_eca_f and not is_last)\n\n    def forward(self, x, x_en):\n        y = x + self.skip_gate(self.skip_conv(x_en))\n        y = self.deconv(self.resblock(y))\n        if not self.is_last:\n            y = self.elu(self.bn(y))\n            y = self.main_eca_f(y)\n        return y\n\n\nclass DeepVQE_Ablation(nn.Module):\n    def __init__(\n        self,\n        prelu_type=None,\n        dw_residual=False,\n        use_eca_f=False,\n        main_block_eca_f=False,\n        gru_hidden=BASE_GRU_HIDDEN,\n        skip_gate=None,\n        dw_subpixel=False,\n        **legacy_kwargs,\n    ):\n        super().__init__()\n        cfg = _normalize_model_config(\n            {\n                "prelu_type": prelu_type,\n                "dw_residual": dw_residual,\n                "use_eca_f": use_eca_f,\n                "main_block_eca_f": main_block_eca_f,\n                "gru_hidden": gru_hidden,\n                "skip_gate": skip_gate,\n                "dw_subpixel": dw_subpixel,\n                **legacy_kwargs,\n            }\n        )\n        self.ablation_config = deepcopy(cfg)\n\n        block_kwargs = {\n            "prelu_type": cfg["prelu_type"],\n            "dw_residual": cfg["dw_residual"],\n            "use_eca_f": cfg["use_eca_f"],\n            "main_block_eca_f": cfg["main_block_eca_f"],\n            "res_groups": cfg["res_groups"],\n        }\n        decoder_kwargs = {\n            **block_kwargs,\n            "skip_gate": cfg["skip_gate"],\n            "dw_subpixel": cfg["dw_subpixel"],\n        }\n\n        self.fe = FE()\n        self.enblock1 = EncoderBlock_Ablation(2, 64, **block_kwargs)\n        self.enblock2 = EncoderBlock_Ablation(64, 128, **block_kwargs)\n        self.enblock3 = EncoderBlock_Ablation(128, 128, **block_kwargs)\n        self.enblock4 = EncoderBlock_Ablation(128, 128, **block_kwargs)\n        self.enblock5 = EncoderBlock_Ablation(128, 128, **block_kwargs)\n\n        self.bottle = Bottleneck_Ablation(128 * 9, int(cfg["gru_hidden"]))\n\n        self.deblock5 = DecoderBlock_Ablation(128, 128, **decoder_kwargs)\n        self.deblock4 = DecoderBlock_Ablation(128, 128, **decoder_kwargs)\n        self.deblock3 = DecoderBlock_Ablation(128, 128, **decoder_kwargs)\n        self.deblock2 = DecoderBlock_Ablation(128, 64, **decoder_kwargs)\n        # Keep the original final activation for Baseline parity, but never\n        # attach main-block ECA-F to the output mask branch.\n        last_kwargs = dict(decoder_kwargs)\n        last_kwargs["main_block_eca_f"] = False\n        self.deblock1 = DecoderBlock_Ablation(64, 27, **last_kwargs)\n        self.ccm = CCM()\n\n    @classmethod\n    def from_config_id(cls, config_id, **overrides):\n        return cls(**get_ablation_config(config_id, **overrides))\n\n    def forward(self, x):\n        """x: (B, F, T, 2)."""\n        en_x0 = self.fe(x)\n        en_x1 = self.enblock1(en_x0)\n        en_x2 = self.enblock2(en_x1)\n        en_x3 = self.enblock3(en_x2)\n        en_x4 = self.enblock4(en_x3)\n        en_x5 = self.enblock5(en_x4)\n\n        en_xr = self.bottle(en_x5)\n\n        de_x5 = self.deblock5(en_xr, en_x5)[..., : en_x4.shape[-1]]\n        de_x4 = self.deblock4(de_x5, en_x4)[..., : en_x3.shape[-1]]\n        de_x3 = self.deblock3(de_x4, en_x3)[..., : en_x2.shape[-1]]\n        de_x2 = self.deblock2(de_x3, en_x2)[..., : en_x1.shape[-1]]\n        de_x1 = self.deblock1(de_x2, en_x1)[..., : en_x0.shape[-1]]\n\n        return self.ccm(de_x1, x)\n\n\nclass StreamResidualBlock_Ablation(nn.Module):\n    def __init__(\n        self,\n        channels,\n        prelu_type=None,\n        dw_residual=False,\n        use_eca_f=False,\n        res_groups=None,\n    ):\n        super().__init__()\n        channels = int(channels)\n        self.dw_residual = bool(dw_residual)\n        self.res_groups = res_groups\n\n        if self.dw_residual:\n            self.depthwise = StreamConv2d(\n                channels,\n                channels,\n                kernel_size=(4, 3),\n                padding=(0, 1),\n                groups=channels,\n            )\n            self.pointwise = nn.Conv2d(channels, channels, kernel_size=1)\n        else:\n            groups = int(res_groups) if res_groups is not None else 1\n            if channels % groups != 0:\n                raise ValueError(f"channels={channels} is not divisible by res_groups={groups}")\n            self.conv = StreamConv2d(\n                channels,\n                channels,\n                kernel_size=(4, 3),\n                padding=(0, 1),\n                groups=groups,\n            )\n\n        self.bn = nn.BatchNorm2d(channels)\n        self.elu = ActivationFactory(prelu_type, channels)\n        self.eca_f = _attention(channels, use_eca_f)\n\n    def forward(self, x, cache):\n        """x: (B, C, 1, F), cache: (B, C, 3, F)."""\n        if self.dw_residual:\n            y, cache = self.depthwise(x, cache)\n            y = self.pointwise(y)\n        else:\n            y, cache = self.conv(x, cache)\n        y = self.eca_f(self.elu(self.bn(y)))\n        return y + x, cache\n\n\nclass StreamEncoderBlock_Ablation(nn.Module):\n    def __init__(\n        self,\n        in_channels,\n        out_channels,\n        kernel_size=(4, 3),\n        stride=(1, 2),\n        prelu_type=None,\n        dw_residual=False,\n        use_eca_f=False,\n        main_block_eca_f=False,\n        res_groups=None,\n    ):\n        super().__init__()\n        self.conv = StreamConv2d(\n            in_channels,\n            out_channels,\n            kernel_size,\n            stride=stride,\n            padding=(0, 1),\n        )\n        self.bn = nn.BatchNorm2d(out_channels)\n        self.elu = ActivationFactory(prelu_type, out_channels)\n        self.main_eca_f = _attention(out_channels, main_block_eca_f)\n        self.resblock = StreamResidualBlock_Ablation(\n            out_channels,\n            prelu_type=prelu_type,\n            dw_residual=dw_residual,\n            use_eca_f=use_eca_f,\n            res_groups=res_groups,\n        )\n\n    def forward(self, x, conv_cache, res_cache):\n        x, conv_cache = self.conv(x, conv_cache)\n        x = self.main_eca_f(self.elu(self.bn(x)))\n        x, res_cache = self.resblock(x, res_cache)\n        return x, conv_cache, res_cache\n\n\nclass StreamBottleneck_Ablation(nn.Module):\n    def __init__(self, input_size, hidden_size):\n        super().__init__()\n        self.gru = nn.GRU(input_size, hidden_size, batch_first=True)\n        self.fc = nn.Linear(hidden_size, input_size)\n\n    def forward(self, x, cache):\n        """x: (B, C, 1, F), cache: (1, B, hidden_size)."""\n        y = rearrange(x, "b c t f -> b t (c f)")\n        y, cache = self.gru(y, cache)\n        y = self.fc(y)\n        y = rearrange(y, "b t (c f) -> b c t f", f=x.shape[-1])\n        return y, cache\n\n\nclass StreamSubpixelConv2d_Ablation(nn.Module):\n    def __init__(self, in_channels, out_channels, kernel_size=(4, 3)):\n        super().__init__()\n        self.conv = StreamConv2d(in_channels, out_channels * 2, kernel_size, padding=(0, 1))\n\n    def forward(self, x, cache):\n        """x: (B, C, 1, F), cache: (B, C, 3, F)."""\n        y, cache = self.conv(x, cache)\n        y = rearrange(y, "b (r c) t f -> b c t (r f)", r=2)\n        return y, cache\n\n\nclass StreamDWSubpixelConv2d_Ablation(nn.Module):\n    def __init__(self, in_channels, out_channels, kernel_size=(4, 3)):\n        super().__init__()\n        self.depthwise = StreamConv2d(\n            in_channels,\n            in_channels,\n            kernel_size,\n            padding=(0, 1),\n            groups=in_channels,\n        )\n        self.pointwise = nn.Conv2d(in_channels, out_channels * 2, kernel_size=1)\n\n    def forward(self, x, cache):\n        """x: (B, C, 1, F), cache: (B, C, 3, F)."""\n        y, cache = self.depthwise(x, cache)\n        y = self.pointwise(y)\n        y = rearrange(y, "b (r c) t f -> b c t (r f)", r=2)\n        return y, cache\n\n\nclass StreamDecoderBlock_Ablation(nn.Module):\n    def __init__(\n        self,\n        in_channels,\n        out_channels,\n        kernel_size=(4, 3),\n        is_last=False,\n        prelu_type=None,\n        dw_residual=False,\n        use_eca_f=False,\n        main_block_eca_f=False,\n        res_groups=None,\n        skip_gate=None,\n        dw_subpixel=False,\n    ):\n        super().__init__()\n        self.skip_conv = nn.Conv2d(in_channels, in_channels, 1)\n        self.skip_gate = _skip_gate(in_channels, skip_gate)\n        self.resblock = StreamResidualBlock_Ablation(\n            in_channels,\n            prelu_type=prelu_type,\n            dw_residual=dw_residual,\n            use_eca_f=use_eca_f,\n            res_groups=res_groups,\n        )\n        if dw_subpixel:\n            self.deconv = StreamDWSubpixelConv2d_Ablation(in_channels, out_channels, kernel_size)\n        else:\n            self.deconv = StreamSubpixelConv2d_Ablation(in_channels, out_channels, kernel_size)\n        self.bn = nn.BatchNorm2d(out_channels)\n        self.elu = ActivationFactory(prelu_type, out_channels)\n        self.is_last = is_last\n        self.main_eca_f = _attention(out_channels, main_block_eca_f and not is_last)\n\n    def forward(self, x, x_en, conv_cache, res_cache):\n        y = x + self.skip_gate(self.skip_conv(x_en))\n        y, res_cache = self.resblock(y, res_cache)\n        y, conv_cache = self.deconv(y, conv_cache)\n        if not self.is_last:\n            y = self.elu(self.bn(y))\n            y = self.main_eca_f(y)\n        return y, conv_cache, res_cache\n\n\nclass StreamCCM_Ablation(nn.Module):\n    """Stateful Complex Convolving Mask block."""\n\n    def __init__(self):\n        super().__init__()\n        self.v = torch.tensor(\n            [[1.0, -0.5, -0.5], [0.0, 0.8660254037844386, -0.8660254037844386]],\n            dtype=torch.float32,\n        )\n        self.unfold = nn.Unfold(kernel_size=(3, 3), padding=(0, 1))\n\n    def forward(self, m, x, cache):\n        """\n        m: (B, 27, 1, F)\n        x: (B, F, 1, 2)\n        cache: (B, F, 2, 2)\n        """\n        m = rearrange(m, "b (r c) t f -> b r c t f", r=3)\n        v = self.v.to(device=m.device, dtype=m.dtype)\n        h_real = torch.sum(v[0][None, :, None, None, None] * m, dim=1)\n        h_imag = torch.sum(v[1][None, :, None, None, None] * m, dim=1)\n\n        m_real = rearrange(h_real, "b (m n) t f -> b m n t f", m=3)\n        m_imag = rearrange(h_imag, "b (m n) t f -> b m n t f", m=3)\n\n        x = torch.cat([cache, x], dim=2)\n        cache = x[:, :, 1:].contiguous()\n        x = x.permute(0, 3, 2, 1).contiguous()\n\n        x_unfold = self.unfold(x)\n        x_unfold = rearrange(\n            x_unfold,\n            "b (c m n) (t f) -> b c m n t f",\n            c=2,\n            m=3,\n            n=3,\n            f=x.shape[-1],\n        )\n\n        x_enh_real = torch.sum(m_real * x_unfold[:, 0] - m_imag * x_unfold[:, 1], dim=(1, 2))\n        x_enh_imag = torch.sum(m_real * x_unfold[:, 1] + m_imag * x_unfold[:, 0], dim=(1, 2))\n        x_enh = torch.stack([x_enh_real, x_enh_imag], dim=3).transpose(1, 2).contiguous()\n        return x_enh, cache\n\n\nclass StreamDeepVQE_Ablation(nn.Module):\n    """Stateful streaming counterpart for every ``DeepVQE_Ablation`` variant."""\n\n    cache_names = (\n        "en_conv_cache1",\n        "en_res_cache1",\n        "en_conv_cache2",\n        "en_res_cache2",\n        "en_conv_cache3",\n        "en_res_cache3",\n        "en_conv_cache4",\n        "en_res_cache4",\n        "en_conv_cache5",\n        "en_res_cache5",\n        "h_cache",\n        "de_conv_cache5",\n        "de_res_cache5",\n        "de_conv_cache4",\n        "de_res_cache4",\n        "de_conv_cache3",\n        "de_res_cache3",\n        "de_conv_cache2",\n        "de_res_cache2",\n        "de_conv_cache1",\n        "de_res_cache1",\n        "m_cache",\n    )\n\n    def __init__(\n        self,\n        prelu_type=None,\n        dw_residual=False,\n        use_eca_f=False,\n        main_block_eca_f=False,\n        gru_hidden=BASE_GRU_HIDDEN,\n        skip_gate=None,\n        dw_subpixel=False,\n        **legacy_kwargs,\n    ):\n        super().__init__()\n        cfg = _normalize_model_config(\n            {\n                "prelu_type": prelu_type,\n                "dw_residual": dw_residual,\n                "use_eca_f": use_eca_f,\n                "main_block_eca_f": main_block_eca_f,\n                "gru_hidden": gru_hidden,\n                "skip_gate": skip_gate,\n                "dw_subpixel": dw_subpixel,\n                **legacy_kwargs,\n            }\n        )\n        self.ablation_config = deepcopy(cfg)\n\n        block_kwargs = {\n            "prelu_type": cfg["prelu_type"],\n            "dw_residual": cfg["dw_residual"],\n            "use_eca_f": cfg["use_eca_f"],\n            "main_block_eca_f": cfg["main_block_eca_f"],\n            "res_groups": cfg["res_groups"],\n        }\n        decoder_kwargs = {\n            **block_kwargs,\n            "skip_gate": cfg["skip_gate"],\n            "dw_subpixel": cfg["dw_subpixel"],\n        }\n\n        self.fe = FE()\n        self.enblock1 = StreamEncoderBlock_Ablation(2, 64, **block_kwargs)\n        self.enblock2 = StreamEncoderBlock_Ablation(64, 128, **block_kwargs)\n        self.enblock3 = StreamEncoderBlock_Ablation(128, 128, **block_kwargs)\n        self.enblock4 = StreamEncoderBlock_Ablation(128, 128, **block_kwargs)\n        self.enblock5 = StreamEncoderBlock_Ablation(128, 128, **block_kwargs)\n\n        self.bottle = StreamBottleneck_Ablation(128 * 9, int(cfg["gru_hidden"]))\n\n        self.deblock5 = StreamDecoderBlock_Ablation(128, 128, **decoder_kwargs)\n        self.deblock4 = StreamDecoderBlock_Ablation(128, 128, **decoder_kwargs)\n        self.deblock3 = StreamDecoderBlock_Ablation(128, 128, **decoder_kwargs)\n        self.deblock2 = StreamDecoderBlock_Ablation(128, 64, **decoder_kwargs)\n        last_kwargs = dict(decoder_kwargs)\n        last_kwargs["main_block_eca_f"] = False\n        self.deblock1 = StreamDecoderBlock_Ablation(64, 27, **last_kwargs)\n        self.ccm = StreamCCM_Ablation()\n\n    @classmethod\n    def from_config_id(cls, config_id, **overrides):\n        return cls(**get_ablation_config(config_id, **overrides))\n\n    @classmethod\n    def from_offline(cls, model, strict=True):\n        stream_model = cls(**model.ablation_config)\n        convert_ablation_to_stream(stream_model, model, strict=strict)\n        stream_model.train(model.training)\n        return stream_model\n\n    def init_cache(self, batch_size=1, freq_bins=257, device=None, dtype=None):\n        """Create zero caches in the fixed order expected by ``forward``."""\n        param = next(self.parameters())\n        device = device if device is not None else param.device\n        dtype = dtype if dtype is not None else param.dtype\n        b = int(batch_size)\n        f0 = int(freq_bins)\n        f1 = (f0 + 1) // 2\n        f2 = (f1 + 1) // 2\n        f3 = (f2 + 1) // 2\n        f4 = (f3 + 1) // 2\n        f5 = (f4 + 1) // 2\n        hidden = self.bottle.gru.hidden_size\n\n        return [\n            torch.zeros(b, 2, 3, f0, device=device, dtype=dtype),\n            torch.zeros(b, 64, 3, f1, device=device, dtype=dtype),\n            torch.zeros(b, 64, 3, f1, device=device, dtype=dtype),\n            torch.zeros(b, 128, 3, f2, device=device, dtype=dtype),\n            torch.zeros(b, 128, 3, f2, device=device, dtype=dtype),\n            torch.zeros(b, 128, 3, f3, device=device, dtype=dtype),\n            torch.zeros(b, 128, 3, f3, device=device, dtype=dtype),\n            torch.zeros(b, 128, 3, f4, device=device, dtype=dtype),\n            torch.zeros(b, 128, 3, f4, device=device, dtype=dtype),\n            torch.zeros(b, 128, 3, f5, device=device, dtype=dtype),\n            torch.zeros(1, b, hidden, device=device, dtype=dtype),\n            torch.zeros(b, 128, 3, f5, device=device, dtype=dtype),\n            torch.zeros(b, 128, 3, f5, device=device, dtype=dtype),\n            torch.zeros(b, 128, 3, f4, device=device, dtype=dtype),\n            torch.zeros(b, 128, 3, f4, device=device, dtype=dtype),\n            torch.zeros(b, 128, 3, f3, device=device, dtype=dtype),\n            torch.zeros(b, 128, 3, f3, device=device, dtype=dtype),\n            torch.zeros(b, 128, 3, f2, device=device, dtype=dtype),\n            torch.zeros(b, 128, 3, f2, device=device, dtype=dtype),\n            torch.zeros(b, 64, 3, f1, device=device, dtype=dtype),\n            torch.zeros(b, 64, 3, f1, device=device, dtype=dtype),\n            torch.zeros(b, f0, 2, 2, device=device, dtype=dtype),\n        ]\n\n    def forward(self, x, cache):\n        """\n        x: (B, F, 1, 2)\n        cache: list of tensors following ``cache_names``.\n        """\n        if len(cache) != len(self.cache_names):\n            raise ValueError(f"Expected {len(self.cache_names)} cache tensors, got {len(cache)}")\n\n        (\n            en_conv_cache1,\n            en_res_cache1,\n            en_conv_cache2,\n            en_res_cache2,\n            en_conv_cache3,\n            en_res_cache3,\n            en_conv_cache4,\n            en_res_cache4,\n            en_conv_cache5,\n            en_res_cache5,\n            h_cache,\n            de_conv_cache5,\n            de_res_cache5,\n            de_conv_cache4,\n            de_res_cache4,\n            de_conv_cache3,\n            de_res_cache3,\n            de_conv_cache2,\n            de_res_cache2,\n            de_conv_cache1,\n            de_res_cache1,\n            m_cache,\n        ) = cache\n\n        en_x0 = self.fe(x)\n        en_x1, en_conv_cache1, en_res_cache1 = self.enblock1(en_x0, en_conv_cache1, en_res_cache1)\n        en_x2, en_conv_cache2, en_res_cache2 = self.enblock2(en_x1, en_conv_cache2, en_res_cache2)\n        en_x3, en_conv_cache3, en_res_cache3 = self.enblock3(en_x2, en_conv_cache3, en_res_cache3)\n        en_x4, en_conv_cache4, en_res_cache4 = self.enblock4(en_x3, en_conv_cache4, en_res_cache4)\n        en_x5, en_conv_cache5, en_res_cache5 = self.enblock5(en_x4, en_conv_cache5, en_res_cache5)\n\n        en_xr, h_cache = self.bottle(en_x5, h_cache)\n\n        de_x5, de_conv_cache5, de_res_cache5 = self.deblock5(en_xr, en_x5, de_conv_cache5, de_res_cache5)\n        de_x5 = de_x5[..., : en_x4.shape[-1]]\n        de_x4, de_conv_cache4, de_res_cache4 = self.deblock4(de_x5, en_x4, de_conv_cache4, de_res_cache4)\n        de_x4 = de_x4[..., : en_x3.shape[-1]]\n        de_x3, de_conv_cache3, de_res_cache3 = self.deblock3(de_x4, en_x3, de_conv_cache3, de_res_cache3)\n        de_x3 = de_x3[..., : en_x2.shape[-1]]\n        de_x2, de_conv_cache2, de_res_cache2 = self.deblock2(de_x3, en_x2, de_conv_cache2, de_res_cache2)\n        de_x2 = de_x2[..., : en_x1.shape[-1]]\n        de_x1, de_conv_cache1, de_res_cache1 = self.deblock1(de_x2, en_x1, de_conv_cache1, de_res_cache1)\n        de_x1 = de_x1[..., : en_x0.shape[-1]]\n\n        x_enh, m_cache = self.ccm(de_x1, x, m_cache)\n\n        new_cache = [\n            en_conv_cache1,\n            en_res_cache1,\n            en_conv_cache2,\n            en_res_cache2,\n            en_conv_cache3,\n            en_res_cache3,\n            en_conv_cache4,\n            en_res_cache4,\n            en_conv_cache5,\n            en_res_cache5,\n            h_cache,\n            de_conv_cache5,\n            de_res_cache5,\n            de_conv_cache4,\n            de_res_cache4,\n            de_conv_cache3,\n            de_res_cache3,\n            de_conv_cache2,\n            de_res_cache2,\n            de_conv_cache1,\n            de_res_cache1,\n            m_cache,\n        ]\n        return x_enh, new_cache\n\n    def forward_flat(self, x, *cache):\n        """ONNX-friendly wrapper: returns ``(enh, *new_cache)``."""\n        y, new_cache = self.forward(x, list(cache))\n        return (y, *new_cache)\n\n\nclass StreamDeepVQE_AblationONNXWrapper(nn.Module):\n    def __init__(self, stream_model):\n        super().__init__()\n        self.stream_model = stream_model\n\n    def forward(self, x, *cache):\n        return self.stream_model.forward_flat(x, *cache)\n\n\ndef convert_ablation_to_stream(stream_model, model, strict=True):\n    """Copy offline ablation weights into the matching stateful stream model."""\n    state_dict = model.state_dict()\n    new_state_dict = stream_model.state_dict()\n    missing = []\n    shape_mismatch = []\n\n    for key, value in new_state_dict.items():\n        candidates = (\n            key,\n            key.replace("Conv2d.", ""),\n            key.replace("conv.Conv2d.", "conv."),\n            key.replace("depthwise.Conv2d.", "depthwise."),\n        )\n        matched = None\n        for candidate in candidates:\n            if candidate in state_dict:\n                matched = candidate\n                break\n        if matched is None:\n            missing.append(key)\n            continue\n        if state_dict[matched].shape != value.shape:\n            shape_mismatch.append((key, matched, tuple(value.shape), tuple(state_dict[matched].shape)))\n            continue\n        new_state_dict[key] = state_dict[matched]\n\n    if strict and (missing or shape_mismatch):\n        details = []\n        if missing:\n            details.append(f"missing={missing}")\n        if shape_mismatch:\n            details.append(f"shape_mismatch={shape_mismatch}")\n        raise ValueError("Unable to convert ablation weights to stream: " + "; ".join(details))\n\n    stream_model.load_state_dict(new_state_dict, strict=False)\n    return stream_model\n\n\n@torch.no_grad()\ndef stream_sequence(stream_model, x, cache=None):\n    """Run a full ``(B, F, T, 2)`` sequence through a stateful stream model."""\n    if cache is None:\n        cache = stream_model.init_cache(x.shape[0], x.shape[1], x.device, x.dtype)\n    outputs = []\n    for frame_idx in range(x.shape[2]):\n        y, cache = stream_model(x[:, :, frame_idx : frame_idx + 1, :], cache)\n        outputs.append(y)\n    return torch.cat(outputs, dim=2), cache\n\n\ndef count_parameters(model):\n    return sum(param.numel() for param in model.parameters())\n', encoding='utf-8')
for module_name in ('ablation.deepvqe_ablation', 'ablation.ablation_config'):
    if module_name in sys.modules:
        del sys.modules[module_name]
importlib.invalidate_caches()

print(f'Đã vào thư mục mã nguồn: {Path.cwd()}')
print(f'Patched {patch_path} with D3_hybrid_head support')
subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], check=False)


## 2.5 Xác nhận ablation code có D3_hybrid_head

Cell này kiểm tra nhanh runtime đã nhận `D3_hybrid_head = Baseline + GRU hidden 768` trước khi khởi tạo model.


In [ ]:
from ablation.deepvqe_ablation import get_ablation_config

d1b_cfg = get_ablation_config('D3_hybrid_head')
print('D3_hybrid_head config OK:', d1b_cfg)
assert d1b_cfg['gru_hidden'] == 768
assert d1b_cfg['use_eca_f'] is False
assert d1b_cfg['dw_residual'] is False
assert d1b_cfg.get('skip_gate') is None


## 3. Tải bộ dữ liệu VoiceBank-DEMAND

Dữ liệu gốc được tải từ các ZIP trên Kaggle Working Dir. Notebook ưu tiên lấy ZIP đã cache trong Kaggle Working Dir, sau đó copy sang Local SSD `/kaggle/working/data` để train nhanh.


In [ ]:
import os
import zipfile
import shutil
from pathlib import Path

# Cache ZIP trên Kaggle Working Dir để lần sau không phải tải lại.
drive_data_dir = WORK_DIR / 'data' / 'voicebank-demand'
drive_data_dir.mkdir(parents=True, exist_ok=True)

# Local SSD của Kaggle để giải nén và train nhanh.
data_dir = Path('/kaggle/working/data/voicebank-demand')
data_dir.mkdir(parents=True, exist_ok=True)

datasets = [
    ('clean_trainset_28spk_wav.zip', 'https://drive.google.com/file/d/1NJr2O4Ik6ueSFlIGSvub8dnFXGTHJ2PG/view?usp=sharing'),
    ('noisy_trainset_28spk_wav.zip', 'https://drive.google.com/file/d/1OqpDIvpVyaTnMbwY1Qt__hfX3X4siMtU/view?usp=sharing'),
    ('clean_testset_wav.zip', 'https://drive.google.com/file/d/1GQc-T1R4FNrhRjTn7AAvAenZTIQEazeH/view?usp=sharing'),
    ('noisy_testset_wav.zip', 'https://drive.google.com/file/d/1rimmCqxXRYRIXZcPkGjQiacr6j1QsMAH/view?usp=sharing'),
]

for filename, url in datasets:
    drive_zip = drive_data_dir / filename
    local_zip = data_dir / filename
    extract_folder = data_dir / filename.replace('.zip', '')

    if drive_zip.exists():
        print(f'Đã tìm thấy {filename} trong Kaggle cache.')
        if not local_zip.exists() and not extract_folder.exists():
            print(f'Copy {filename} từ Kaggle cache sang Local SSD...')
            shutil.copy2(drive_zip, local_zip)
    else:
        print(f'Không thấy {filename} trong Kaggle cache; tải xuống Local SSD...')
        if not extract_folder.exists():
            if local_zip.exists() and not zipfile.is_zipfile(local_zip):
                print(f'File {local_zip} bị lỗi, đang xóa...')
                local_zip.unlink()
            if not local_zip.exists():
                os.system(f'gdown --fuzzy {url} -O {local_zip}')
            if local_zip.exists() and zipfile.is_zipfile(local_zip):
                print(f'Lưu cache {filename} vào Kaggle cache...')
                shutil.copy2(local_zip, drive_zip)

    if not extract_folder.exists():
        if local_zip.exists():
            print(f'Đang giải nén {filename}...')
            with zipfile.ZipFile(local_zip, 'r') as zip_ref:
                zip_ref.extractall(data_dir)
            print(f'Hoàn tất giải nén {filename}.')
        else:
            print(f'Lỗi: Không tìm thấy file {local_zip} để giải nén!')

print('Dữ liệu đã sẵn sàng trên Local SSD của Kaggle.')


## 4. Xử lý và phân chia Dataset (Tạo file CSV)

Tạo `train.csv`, `valid.csv`, `test.csv` giống baseline v4: train/valid split 90/10 với `random_state=42`.


In [ ]:
import glob
import pandas as pd
from pathlib import Path

data_dir = Path(data_dir)


def create_csv(clean_dir, noisy_dir, output_csv):
    clean_files = sorted(glob.glob(str(Path(clean_dir) / '*.wav')))
    noisy_files = sorted(glob.glob(str(Path(noisy_dir) / '*.wav')))

    assert len(clean_files) == len(noisy_files), f'Số lượng file không khớp! ({len(clean_files)} != {len(noisy_files)})'
    mismatched = [
        (Path(c).name, Path(n).name)
        for c, n in zip(clean_files, noisy_files)
        if Path(c).name != Path(n).name
    ]
    assert not mismatched, f'Tên file clean/noisy không khớp: {mismatched[:5]}'

    rows = []
    for clean, noisy in zip(clean_files, noisy_files):
        filename = Path(clean).name
        rows.append({
            'ID': filename.replace('.wav', ''),
            'clean_wav': str(Path(clean).resolve()),
            'noisy_wav': str(Path(noisy).resolve()),
        })

    df = pd.DataFrame(rows)
    df.to_csv(output_csv, index=False)
    print(f'Đã tạo {output_csv} với {len(df)} mẫu.')


create_csv(data_dir / 'clean_trainset_28spk_wav', data_dir / 'noisy_trainset_28spk_wav', data_dir / 'train_full.csv')
create_csv(data_dir / 'clean_testset_wav', data_dir / 'noisy_testset_wav', data_dir / 'test.csv')

train_full = pd.read_csv(data_dir / 'train_full.csv')
train_full = train_full.sample(frac=1, random_state=42).reset_index(drop=True)
split_idx = int(len(train_full) * 0.90)
train_full.iloc[:split_idx].to_csv(data_dir / 'train.csv', index=False)
train_full.iloc[split_idx:].to_csv(data_dir / 'valid.csv', index=False)

print(f'Train: {split_idx} | Valid: {len(train_full) - split_idx}')
print(f'Test: {len(pd.read_csv(data_dir / "test.csv"))}')
print('Hoàn tất tạo metadata CSV!')


## 5. Cấu hình Hyperparameters

Các giá trị bám baseline v4: seed `1234`, STFT `sqrt_hann`, batch size `8`, optimizer Adam, loss RI/Mag/SI-SNR. Notebook chỉ train `D3_hybrid_head`.


In [ ]:
import json
from pathlib import Path

data_dir = Path(data_dir)
WORK_DIR = Path(WORK_DIR)

CONFIG = {
    'config_id': 'D3_hybrid_head',
    'seed': 1234,
    'sample_rate': 16000,
    'n_fft': 512,
    'hop_length': 256,
    'win_length': 512,
    'stft_window': 'sqrt_hann',

    'batch_size': 8,
    'valid_batch_size': 4,
    'grad_accum_steps': 1,
    'num_workers': 2,
    'persistent_workers': True,
    'prefetch_factor': 2,
    'progress_update_every': 10,
    'epochs': 80,
    'lr': 1e-3,
    'weight_decay': 0.0,
    'grad_clip': 5.0,

    'lamda_ri': 30.0,
    'lamda_mag': 70.0,
    'compress_factor': 0.3,

    'train_csv': str(data_dir / 'train.csv'),
    'valid_csv': str(data_dir / 'valid.csv'),
    'test_csv': str(data_dir / 'test.csv'),

    'checkpoint_dir': str(WORK_DIR / 'runs' / 'ablation' / 'D3_hybrid_head_scratch_v1'),
    'output_dir': str(WORK_DIR / 'runs' / 'ablation' / 'D3_hybrid_head_scratch_v1'),
    'resume_checkpoint': 'last.pt',
    'resume_existing': False,
    'results_dir': str(WORK_DIR / 'results' / 'ablation'),
    'onnx_dir': str(WORK_DIR / 'onnx_models' / 'ablation'),

    'scheduler_factor': 0.5,
    'scheduler_patience': 5,
    'scheduler_min_lr': 1e-6,
    'early_stopping_patience': 15,
    'early_stopping_min_delta': 0.0,
    'early_stopping_min_epochs': 0,

    'use_amp': True,
    'augment': True,
    'aug_gain_range_db': (-6.0, 6.0),
    'aug_snr_remix_range': (0.0, 20.0),
    'aug_prob': 0.5,

    # True để khớp protocol hiện tại. Khi chốt phương án và train sạch, đổi thành False cho mọi variant được so sánh.
    'valid_random_crop': True,

    'tensorboard': False,
    'use_wandb': True,
    'wandb_project': 'DeepVQE-Phase2-D3-GRU768',
    'eval_pesq_every': 5,
    'eval_pesq_samples': 50,
    'log_audio_every': 0,

    'run_training': True,
    'run_smoke_tests': True,
    'run_arch_benchmark': True,
    'run_quality_eval': True,
    'run_onnx_export': False,
}

for key in ['checkpoint_dir', 'output_dir', 'results_dir', 'onnx_dir']:
    Path(CONFIG[key]).mkdir(parents=True, exist_ok=True)

print(json.dumps(CONFIG, indent=2, default=str))


## 6. Dataset & DataLoader (Pure PyTorch)

Cùng kiểu baseline v4: crop đoạn 3 giây để tiết kiệm VRAM, augment chỉ áp dụng cho training set.


In [ ]:
import random
import numpy as np
import torch
import torchaudio
from torch.utils.data import Dataset, DataLoader


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything(CONFIG['seed'])


def repeat_to_length(wav, length):
    if wav.numel() == 0:
        raise ValueError('Encountered empty waveform')
    if wav.numel() >= length:
        return wav
    repeats = int(np.ceil(length / wav.numel()))
    return wav.repeat(repeats)[:length]


def crop_pair(noisy, clean, segment_len, random_crop=True):
    min_len = min(noisy.shape[0], clean.shape[0])
    noisy = noisy[:min_len]
    clean = clean[:min_len]

    if segment_len is None:
        return noisy, clean
    if min_len <= segment_len:
        return noisy, clean

    if random_crop:
        start = random.randint(0, min_len - segment_len)
    else:
        start = max(0, (min_len - segment_len) // 2)
    return noisy[start:start + segment_len], clean[start:start + segment_len]


class VCTKDemandDataset(Dataset):
    """Dataset cho VCTK-DEMAND: đọc cặp (noisy, clean) từ CSV."""

    def __init__(self, csv_path, sample_rate=16000, segment_len=None, augment_cfg=None, random_crop=True):
        self.df = pd.read_csv(csv_path)
        self.sample_rate = sample_rate
        self.segment_len = segment_len
        self.aug = augment_cfg
        self.random_crop = random_crop

    def __len__(self):
        return len(self.df)

    def _load(self, path):
        wav, sr = torchaudio.load(path)
        wav = wav.float()
        if wav.shape[0] > 1:
            wav = wav.mean(dim=0, keepdim=True)
        if sr != self.sample_rate:
            wav = torchaudio.functional.resample(wav, sr, self.sample_rate)
        return wav.squeeze(0)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        noisy = self._load(row['noisy_wav'])
        clean = self._load(row['clean_wav'])
        noisy, clean = crop_pair(noisy, clean, self.segment_len, random_crop=self.random_crop)

        if self.aug is not None:
            aug_prob = self.aug.get('aug_prob', 0.5)

            if random.random() < aug_prob:
                lo, hi = self.aug['aug_gain_range_db']
                gain_db = random.uniform(lo, hi)
                gain = 10 ** (gain_db / 20.0)
                noisy = noisy * gain
                clean = clean * gain

            if random.random() < aug_prob:
                noise = noisy - clean
                lo, hi = self.aug['aug_snr_remix_range']
                target_snr = random.uniform(lo, hi)
                clean_rms = clean.pow(2).mean().sqrt().clamp(min=1e-8)
                noise_rms = noise.pow(2).mean().sqrt().clamp(min=1e-8)
                target_noise_rms = clean_rms / (10 ** (target_snr / 20.0))
                noisy = clean + noise * (target_noise_rms / noise_rms)

        return {'noisy': noisy, 'clean': clean, 'id': row['ID']}


def collate_fn(batch):
    max_len = max(item['noisy'].shape[0] for item in batch)
    noisy_batch, clean_batch, ids = [], [], []
    for item in batch:
        noisy, clean = item['noisy'], item['clean']
        pad_len = max_len - noisy.shape[0]
        if pad_len > 0:
            noisy = torch.nn.functional.pad(noisy, (0, pad_len))
            clean = torch.nn.functional.pad(clean, (0, pad_len))
        noisy_batch.append(noisy)
        clean_batch.append(clean)
        ids.append(item['id'])

    return {
        'noisy': torch.stack(noisy_batch),
        'clean': torch.stack(clean_batch),
        'id': ids,
    }


segment_len = int(3.0 * CONFIG['sample_rate'])
aug_cfg = {k: v for k, v in CONFIG.items() if k.startswith('aug_')} if CONFIG.get('augment') else None

train_dataset = VCTKDemandDataset(
    CONFIG['train_csv'],
    CONFIG['sample_rate'],
    segment_len=segment_len,
    augment_cfg=aug_cfg,
    random_crop=True,
)
valid_dataset = VCTKDemandDataset(
    CONFIG['valid_csv'],
    CONFIG['sample_rate'],
    segment_len=segment_len,
    augment_cfg=None,
    random_crop=CONFIG.get('valid_random_crop', True),
)

loader_kwargs = dict(
    num_workers=CONFIG['num_workers'],
    pin_memory=torch.cuda.is_available(),
    collate_fn=collate_fn,
)
if CONFIG['num_workers'] > 0:
    loader_kwargs.update(
        persistent_workers=CONFIG.get('persistent_workers', False),
        prefetch_factor=CONFIG.get('prefetch_factor', 2),
    )

train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=True,
    drop_last=True,
    **loader_kwargs,
)
valid_loader = DataLoader(
    valid_dataset,
    batch_size=CONFIG.get('valid_batch_size', CONFIG['batch_size']),
    shuffle=False,
    drop_last=False,
    **loader_kwargs,
)

print(f'Train: {len(train_dataset)} samples, {len(train_loader)} batches')
print(f'Valid: {len(valid_dataset)} samples, {len(valid_loader)} batches')
print(f'Valid crop: {"random như baseline v4" if CONFIG.get("valid_random_crop", True) else "center/fixed"}')
if aug_cfg:
    print(f'Augmentation: ON (gain={aug_cfg["aug_gain_range_db"]}, snr_remix={aug_cfg["aug_snr_remix_range"]}, prob={aug_cfg["aug_prob"]})')
else:
    print('Augmentation: OFF')


## 7. Khởi tạo Model D3_hybrid_head, Optimizer, Scheduler

Khác baseline gốc ở đúng một điểm: model dùng `DeepVQE_Ablation.from_config_id('D3_hybrid_head')`, tức tăng bottleneck GRU hidden từ 576 lên 768 và giữ nguyên encoder/decoder/residual/CCM.


In [ ]:
import time
import json
from pathlib import Path
from torch.utils.tensorboard import SummaryWriter

from ablation.ablation_config import deep_update, get_train_config, reproducibility_metadata
from ablation.deepvqe_ablation import DeepVQE_Ablation, count_parameters


def make_grad_scaler(enabled):
    if hasattr(torch, 'amp') and hasattr(torch.amp, 'GradScaler'):
        return torch.amp.GradScaler('cuda', enabled=enabled)
    return torch.cuda.amp.GradScaler(enabled=enabled)


def autocast_context(device, enabled):
    if hasattr(torch, 'amp') and hasattr(torch.amp, 'autocast'):
        return torch.amp.autocast(device.type, enabled=enabled)
    return torch.cuda.amp.autocast(enabled=enabled)


def build_train_cfg(config):
    override = {
        'experiment': {
            'config_id': config['config_id'],
            'name': f"deepvqe_ablation_{config['config_id']}",
            'seed': config['seed'],
            'output_dir': config['output_dir'],
            'resume_from': None,
        },
        'stft': {
            'n_fft': config['n_fft'],
            'hop_length': config['hop_length'],
            'win_length': config['win_length'],
            'window': config['stft_window'],
        },
        'data': {
            'sample_rate': config['sample_rate'],
            'clip_seconds': 3.0,
            'num_workers': config['num_workers'],
            'pin_memory': torch.cuda.is_available(),
            'train_manifest': config['train_csv'],
            'valid_manifest': config['valid_csv'],
            'test_manifest': config['test_csv'],
            'augment': config['augment'],
            'aug_gain_range_db': list(config['aug_gain_range_db']),
            'aug_snr_remix_range': list(config['aug_snr_remix_range']),
            'aug_prob': config['aug_prob'],
        },
        'optimizer': {
            'lr': config['lr'],
            'weight_decay': config['weight_decay'],
            'betas': [0.9, 0.999],
            'grad_clip_norm': config['grad_clip'],
        },
        'scheduler': {
            'mode': 'min',
            'factor': config['scheduler_factor'],
            'patience': config['scheduler_patience'],
            'min_lr': config['scheduler_min_lr'],
        },
        'training': {
            'device': 'cuda' if torch.cuda.is_available() else 'cpu',
            'batch_size': config['batch_size'],
            'valid_batch_size': config['valid_batch_size'],
            'epochs': config['epochs'],
            'grad_accum_steps': config['grad_accum_steps'],
            'checkpoint_monitor': 'loss',
            'checkpoint_mode': 'min',
            'use_amp': config['use_amp'],
            'early_stop_patience': config['early_stopping_patience'],
            'early_stop_min_delta': config['early_stopping_min_delta'],
            'early_stop_min_epochs': config['early_stopping_min_epochs'],
            'valid_random_crop': config['valid_random_crop'],
        },
        'loss': {
            'lamda_ri': config['lamda_ri'],
            'lamda_mag': config['lamda_mag'],
            'compress_factor': config['compress_factor'],
        },
        'logging': {
            'use_wandb': config['use_wandb'],
            'wandb_project': config['wandb_project'],
            'eval_pesq_every': config['eval_pesq_every'],
            'eval_pesq_samples': config['eval_pesq_samples'],
            'log_audio_every': config['log_audio_every'],
        },
    }
    return deep_update(get_train_config(config['config_id']), override)


TRAIN_CFG = build_train_cfg(CONFIG)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU count: {torch.cuda.device_count()}')
    for i in range(torch.cuda.device_count()):
        print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')

# D3 uses Baseline architecture with a larger bottleneck GRU hidden size: 576 -> 768.
model = DeepVQE_Ablation.from_config_id(CONFIG['config_id']).to(device)
USE_DATA_PARALLEL = False
if USE_DATA_PARALLEL and torch.cuda.device_count() > 1:
    print(f'Bật DataParallel trên {torch.cuda.device_count()} GPUs')
    model = torch.nn.DataParallel(model)
else:
    print('DataParallel: OFF')

total_params = count_parameters(model.module if hasattr(model, 'module') else model)
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params: {total_params / 1e6:.2f}M | Trainable: {trainable_params / 1e6:.2f}M')

optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay'])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=CONFIG['scheduler_factor'],
    patience=CONFIG['scheduler_patience'],
    min_lr=CONFIG['scheduler_min_lr'],
)

use_amp = CONFIG['use_amp'] and device.type == 'cuda'
scaler = make_grad_scaler(enabled=use_amp)
print(f'Mixed Precision (AMP): {"ON" if use_amp else "OFF"}')

window = torch.hann_window(CONFIG['win_length']).sqrt().to(device)

output_dir = Path(CONFIG.get('checkpoint_dir', CONFIG['output_dir']))
output_dir.mkdir(parents=True, exist_ok=True)
Path(CONFIG['results_dir']).mkdir(parents=True, exist_ok=True)
Path(CONFIG['onnx_dir']).mkdir(parents=True, exist_ok=True)

with open(output_dir / 'config.json', 'w', encoding='utf-8') as f:
    json.dump(TRAIN_CFG, f, indent=2, default=str)
with open(output_dir / 'notebook_config.json', 'w', encoding='utf-8') as f:
    json.dump(CONFIG, f, indent=2, default=str)

tb_log_dir = output_dir / 'tb_logs'
tb_writer = SummaryWriter(log_dir=str(tb_log_dir)) if CONFIG.get('tensorboard', False) else None
print(f'Output dir: {output_dir}')
print(f'TensorBoard: {"ON" if tb_writer else "OFF"}')


## 8. Hàm tiện ích: STFT, Loss, Checkpoint, Log, Metrics

Giữ cùng loss baseline v4: compressed RI + compressed magnitude + negative SI-SNR trên waveform gốc.


In [ ]:
import csv
import shutil


def make_stft(wav, n_fft, hop_length, win_length, win):
    spec = torch.stft(
        wav,
        n_fft=n_fft,
        hop_length=hop_length,
        win_length=win_length,
        window=win,
        return_complex=True,
    )
    return torch.view_as_real(spec)


def make_istft(spec, n_fft, hop_length, win_length, win, length=None):
    complex_spec = torch.complex(spec[..., 0], spec[..., 1])
    return torch.istft(
        complex_spec,
        n_fft=n_fft,
        hop_length=hop_length,
        win_length=win_length,
        window=win,
        length=length,
    )


def compute_loss(model, noisy_wav, clean_wav, cfg, win):
    n_fft = cfg['n_fft']
    hop = cfg['hop_length']
    win_len = cfg['win_length']
    c = cfg['compress_factor']

    noisy_spec = make_stft(noisy_wav, n_fft, hop, win_len, win)

    amp_enabled = cfg.get('use_amp', False) and noisy_wav.device.type == 'cuda'
    with autocast_context(noisy_wav.device, enabled=amp_enabled):
        pred_stft = model(noisy_spec)

    pred_stft = pred_stft.float()
    clean_spec = make_stft(clean_wav, n_fft, hop, win_len, win)

    min_t = min(pred_stft.shape[2], clean_spec.shape[2])
    pred_stft = pred_stft[:, :, :min_t, :]
    true_stft = clean_spec[:, :, :min_t, :]

    pred_real, pred_imag = pred_stft[..., 0], pred_stft[..., 1]
    true_real, true_imag = true_stft[..., 0], true_stft[..., 1]

    pred_mag = torch.sqrt(pred_real**2 + pred_imag**2 + 1e-12)
    true_mag = torch.sqrt(true_real**2 + true_imag**2 + 1e-12)

    pred_real_c = pred_real / (pred_mag**(1 - c))
    pred_imag_c = pred_imag / (pred_mag**(1 - c))
    true_real_c = true_real / (true_mag**(1 - c))
    true_imag_c = true_imag / (true_mag**(1 - c))

    real_loss = torch.mean((pred_real_c - true_real_c)**2)
    imag_loss = torch.mean((pred_imag_c - true_imag_c)**2)
    mag_loss = torch.mean((pred_mag**c - true_mag**c)**2)

    y_pred = torch.istft(
        pred_real + 1j * pred_imag,
        n_fft=n_fft,
        hop_length=hop,
        win_length=win_len,
        window=win,
    )
    y_true = clean_wav
    min_wav_len = min(y_pred.shape[-1], y_true.shape[-1])
    y_pred = y_pred[..., :min_wav_len]
    y_true = y_true[..., :min_wav_len]

    eps = 1e-8
    true_energy = torch.sum(torch.square(y_true), dim=-1, keepdim=True)
    y_target = torch.sum(y_true * y_pred, dim=-1, keepdim=True) * y_true / (true_energy + eps)
    target_energy = torch.sum(torch.square(y_target), dim=-1, keepdim=True)
    noise_energy = torch.sum(torch.square(y_pred - y_target), dim=-1, keepdim=True)
    sisnr = -10 * torch.log10((target_energy + eps) / (noise_energy + eps)).mean()

    loss = cfg['lamda_ri'] * (real_loss + imag_loss) + cfg['lamda_mag'] * mag_loss + sisnr
    return loss, {
        'loss': loss.item(),
        'ri_loss': (real_loss + imag_loss).item(),
        'mag_loss': mag_loss.item(),
        'sisnr': sisnr.item(),
    }


def unwrap_model(model):
    return model.module if hasattr(model, 'module') else model


def save_checkpoint(path, model, optimizer, scheduler, epoch, best_loss, bad_epochs, scaler=None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    model_state = unwrap_model(model).state_dict()
    metadata = reproducibility_metadata(TRAIN_CFG, checkpoint_id=path.stem)
    ckpt = {
        'model': model_state,
        'optimizer': optimizer.state_dict(),
        'scheduler': scheduler.state_dict() if scheduler is not None else None,
        'scaler': scaler.state_dict() if scaler is not None else None,
        'epoch': epoch,
        'best_loss': best_loss,
        'best_metric': best_loss,
        'bad_epochs': bad_epochs,
        'config': TRAIN_CFG,
        'metadata': metadata,
    }

    temp_local = Path('/kaggle/working') / f'temp_{path.name}' if Path('/kaggle/working').exists() else Path(f'temp_{path.name}')
    temp_target = path.with_suffix(path.suffix + '.tmp')
    torch.save(ckpt, temp_local)
    shutil.copy2(temp_local, temp_target)
    os.replace(temp_target, path)
    if temp_local.exists():
        temp_local.unlink()


def load_checkpoint(path, model, optimizer=None, scheduler=None, device='cpu', scaler=None):
    ckpt = torch.load(str(path), map_location=device, weights_only=False)
    unwrap_model(model).load_state_dict(ckpt['model'])
    if optimizer is not None and ckpt.get('optimizer'):
        optimizer.load_state_dict(ckpt['optimizer'])
    if scheduler is not None and ckpt.get('scheduler'):
        scheduler.load_state_dict(ckpt['scheduler'])
    if scaler is not None and ckpt.get('scaler'):
        scaler.load_state_dict(ckpt['scaler'])
    return ckpt.get('epoch', 0), ckpt.get('best_loss', ckpt.get('best_metric')), ckpt.get('bad_epochs', 0)


def set_optimizer_lr(optimizer, lr):
    for group in optimizer.param_groups:
        group['lr'] = lr


def optimizer_step_with_amp(optimizer, scaler, model, grad_clip):
    if grad_clip:
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
    scaler.step(optimizer)
    scaler.update()
    optimizer.zero_grad(set_to_none=True)


def append_log(log_path, row_dict):
    log_path = Path(log_path)
    file_exists = log_path.exists() and log_path.stat().st_size > 0
    with open(log_path, 'a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=list(row_dict.keys()))
        if not file_exists:
            writer.writeheader()
        writer.writerow(row_dict)


def compute_pesq_stoi(model, dataloader, cfg, win, device, max_samples=50):
    from pesq import pesq
    from pystoi import stoi

    m = unwrap_model(model)
    was_training = m.training
    m.eval()
    sr = cfg['sample_rate']
    pesq_scores, stoi_scores = [], []
    count = 0
    amp_enabled = cfg.get('use_amp', False) and device.type == 'cuda'

    with torch.no_grad():
        for batch in dataloader:
            noisy = batch['noisy'].to(device)
            clean = batch['clean']
            spec = make_stft(noisy, cfg['n_fft'], cfg['hop_length'], cfg['win_length'], win)
            with autocast_context(device, enabled=amp_enabled):
                pred = model(spec)
            enhanced = make_istft(pred.float(), cfg['n_fft'], cfg['hop_length'], cfg['win_length'], win, length=noisy.shape[-1])

            for i in range(enhanced.shape[0]):
                enh_np = enhanced[i].detach().cpu().numpy()
                cln_np = clean[i].numpy()
                min_l = min(len(enh_np), len(cln_np))
                enh_np, cln_np = enh_np[:min_l], cln_np[:min_l]
                try:
                    pesq_scores.append(pesq(sr, cln_np, enh_np, 'wb'))
                except Exception:
                    pass
                try:
                    stoi_scores.append(stoi(cln_np, enh_np, sr, extended=False))
                except Exception:
                    pass
                count += 1
                if count >= max_samples:
                    break
            if count >= max_samples:
                break

    m.train(was_training)
    return {
        'pesq': float(np.mean(pesq_scores)) if pesq_scores else 0.0,
        'stoi': float(np.mean(stoi_scores)) if stoi_scores else 0.0,
    }


print('Hàm tiện ích đã sẵn sàng.')


## 8.5 Khởi động TensorBoard (tùy chọn)


In [ ]:
if CONFIG.get('tensorboard', False):
    ip = get_ipython()
    ip.run_line_magic('load_ext', 'tensorboard')
    ip.run_line_magic('tensorboard', f'--logdir {str(output_dir / "tb_logs")}')
else:
    print('TensorBoard đang tắt trong CONFIG để ưu tiên tốc độ train.')


## 8.6 Đăng nhập WandB (tùy chọn)


In [ ]:
if CONFIG.get('use_wandb', False):
    import os
    import wandb

    wandb_key = os.environ.get('WANDB_API_KEY')
    if not wandb_key:
        try:
            from kaggle_secrets import UserSecretsClient
            wandb_key = UserSecretsClient().get_secret('WANDB_API_KEY')
        except Exception as exc:
            print(f'Không lấy được WANDB_API_KEY từ Kaggle Secrets/env: {exc}')

    if wandb_key:
        wandb.login(key=wandb_key, relogin=True)
        print('Đã đăng nhập WandB bằng WANDB_API_KEY.')
    else:
        print('Chưa có WANDB_API_KEY; wandb.init có thể yêu cầu đăng nhập thủ công.')
else:
    print('WandB đang tắt trong CONFIG. Đổi use_wandb=True nếu muốn log lên WandB.')


## 9. Vòng lặp huấn luyện D3_hybrid_head (Training Loop)

D3_hybrid_head mặc định `resume_existing=False` để lần chạy đầu train từ scratch. Nếu Kaggle session bị ngắt sau khi D3 đã tạo checkpoint hợp lệ, đổi `resume_existing=True` trong CONFIG để tiếp tục từ `D3_hybrid_head_scratch_v1/last.pt`.


In [ ]:
try:
    from tqdm.notebook import tqdm
except ImportError:
    from tqdm import tqdm

wandb = None
if CONFIG.get('use_wandb', False):
    import wandb as _wandb
    wandb = _wandb
    wandb.init(project=CONFIG.get('wandb_project', 'DeepVQE-Ablation'), config=CONFIG, resume='allow')
    _watch_target = model.module if hasattr(model, 'module') else model
    wandb.watch(_watch_target, log='gradients', log_freq=100)

start_epoch = 0
best_loss = None
bad_epochs = 0

resume_path = output_dir / CONFIG.get('resume_checkpoint', 'last.pt')
resume_existing = bool(CONFIG.get('resume_existing', True))
if resume_existing and resume_path.exists():
    start_epoch, best_loss, bad_epochs = load_checkpoint(
        resume_path,
        model,
        optimizer,
        scheduler,
        device=device,
        scaler=scaler,
    )
    resumed_lr = optimizer.param_groups[0]['lr']
    effective_lr = min(resumed_lr, CONFIG['lr'])
    set_optimizer_lr(optimizer, effective_lr)
    _best_str = f'{best_loss:.6f}' if best_loss is not None else 'N/A'
    print(f'Đã resume từ epoch {start_epoch}, best_loss={_best_str}, bad_epochs={bad_epochs}')
    print(f'LR checkpoint: {resumed_lr:.2e} | LR config: {CONFIG["lr"]:.2e} -> LR dùng: {effective_lr:.2e}')
else:
    if resume_path.exists() and not resume_existing:
        print(f'D3_hybrid_head train-from-scratch: bỏ qua checkpoint cũ {resume_path}')
    print('Bắt đầu training từ đầu.')
    print(f'LR config: {optimizer.param_groups[0]["lr"]:.2e}')

log_path = output_dir / 'train_log.csv'
print(f'Log file: {log_path}')
print(f'AMP: {"ON" if use_amp else "OFF"} | Batch: {CONFIG["batch_size"]} | Accum: {CONFIG["grad_accum_steps"]}')

if CONFIG.get('run_training', True):
    accum_steps = CONFIG.get('grad_accum_steps', 1)
    progress_update_every = max(1, CONFIG.get('progress_update_every', 10))

    for epoch in range(start_epoch + 1, CONFIG['epochs'] + 1):
        model.train()
        train_losses = []
        skipped_train_batches = 0
        valid_accum_batches = 0
        t0 = time.time()
        optimizer.zero_grad(set_to_none=True)

        pbar = tqdm(train_loader, desc=f'Epoch {epoch} [Train]', leave=False)
        for batch_idx, batch in enumerate(pbar):
            noisy = batch['noisy'].to(device, non_blocking=True)
            clean = batch['clean'].to(device, non_blocking=True)

            loss, parts = compute_loss(model, noisy, clean, CONFIG, window)
            if not torch.isfinite(loss):
                skipped_train_batches += 1
                optimizer.zero_grad(set_to_none=True)
                print(f'  [WARN] Skip train batch {batch_idx}: non-finite loss | ids={batch["id"][:3]}')
                continue

            scaler.scale(loss / accum_steps).backward()
            valid_accum_batches += 1
            if valid_accum_batches % accum_steps == 0:
                optimizer_step_with_amp(optimizer, scaler, model, CONFIG['grad_clip'])

            train_losses.append(parts)
            if batch_idx % progress_update_every == 0 or (batch_idx + 1) == len(train_loader):
                pbar.set_postfix({'loss': f"{parts['loss']:.4f}"})

        if valid_accum_batches % accum_steps != 0:
            optimizer_step_with_amp(optimizer, scaler, model, CONFIG['grad_clip'])
        if not train_losses:
            print('Không có train batch hợp lệ trong epoch này; dừng training để tránh ghi checkpoint lỗi.')
            break
        avg_train = {k: float(np.mean([d[k] for d in train_losses])) for k in train_losses[0]}
        elapsed = time.time() - t0

        model.eval()
        valid_losses = []
        skipped_valid_batches = 0
        with torch.no_grad():
            for batch in tqdm(valid_loader, desc=f'Epoch {epoch} [Valid]', leave=False):
                noisy = batch['noisy'].to(device, non_blocking=True)
                clean = batch['clean'].to(device, non_blocking=True)
                valid_loss, parts = compute_loss(model, noisy, clean, CONFIG, window)
                if not torch.isfinite(valid_loss):
                    skipped_valid_batches += 1
                    print(f'  [WARN] Skip valid batch: non-finite loss | ids={batch["id"][:3]}')
                    continue
                valid_losses.append(parts)

        if not valid_losses:
            print('Không có valid batch hợp lệ trong epoch này; dừng training để tránh ghi checkpoint lỗi.')
            break
        avg_valid = {k: float(np.mean([d[k] for d in valid_losses])) for k in valid_losses[0]}
        train_has_nan = skipped_train_batches > 0 or not np.isfinite(avg_train['loss'])
        valid_has_nan = skipped_valid_batches > 0 or not np.isfinite(avg_valid['loss'])

        pesq_stoi = {'pesq': 0.0, 'stoi': 0.0}
        eval_every = CONFIG.get('eval_pesq_every', 0)
        if eval_every > 0 and epoch % eval_every == 0:
            print(f'  Đang tính PESQ/STOI trên valid subset (mỗi {eval_every} epoch)...')
            pesq_stoi = compute_pesq_stoi(
                model,
                valid_loader,
                CONFIG,
                window,
                device,
                max_samples=CONFIG.get('eval_pesq_samples', 50),
            )
            print(f'  >> PESQ: {pesq_stoi["pesq"]:.3f} | STOI: {pesq_stoi["stoi"]:.4f}')

        if train_has_nan or valid_has_nan:
            print(f'  [WARN] Epoch {epoch} có batch non-finite: train_skipped={skipped_train_batches}, valid_skipped={skipped_valid_batches}.')
            print('  [WARN] Bỏ qua scheduler/checkpoint epoch này. Training vẫn tiếp tục...')
            continue

        prev_lr = optimizer.param_groups[0]['lr']
        scheduler.step(avg_valid['loss'])
        current_lr = optimizer.param_groups[0]['lr']
        scheduler_bad_epochs = getattr(scheduler, 'num_bad_epochs', None)
        if current_lr < prev_lr:
            print(f'  >>> Scheduler giảm LR: {prev_lr:.2e} -> {current_lr:.2e}')

        previous_best = best_loss
        is_best = previous_best is None or avg_valid['loss'] < previous_best
        improved_enough = previous_best is None or avg_valid['loss'] < previous_best - CONFIG.get('early_stopping_min_delta', 0.0)
        if is_best:
            best_loss = avg_valid['loss']
        if improved_enough:
            bad_epochs = 0
        else:
            bad_epochs += 1

        save_checkpoint(output_dir / 'last.pt', model, optimizer, scheduler, epoch, best_loss, bad_epochs, scaler=scaler)
        if is_best:
            save_checkpoint(output_dir / 'best.pt', model, optimizer, scheduler, epoch, best_loss, bad_epochs, scaler=scaler)
            print(f'  >>> Saved best model (valid_loss={best_loss:.6f})')

        print(
            f"Epoch {epoch:3d}/{CONFIG['epochs']} | "
            f"Train Loss: {avg_train['loss']:.6f} (ri={avg_train['ri_loss']:.4f}, mag={avg_train['mag_loss']:.4f}, sisnr={avg_train['sisnr']:.4f}) | "
            f"Valid Loss: {avg_valid['loss']:.6f} (ri={avg_valid['ri_loss']:.4f}, mag={avg_valid['mag_loss']:.4f}, sisnr={avg_valid['sisnr']:.4f}) | "
            f"LR: {current_lr:.2e} | Sched bad: {scheduler_bad_epochs}/{CONFIG['scheduler_patience']} | Time: {elapsed:.0f}s"
        )

        append_log(log_path, {
            'epoch': epoch,
            'train_loss': f"{avg_train['loss']:.6f}",
            'train_ri_loss': f"{avg_train['ri_loss']:.6f}",
            'train_mag_loss': f"{avg_train['mag_loss']:.6f}",
            'train_sisnr': f"{avg_train['sisnr']:.6f}",
            'valid_loss': f"{avg_valid['loss']:.6f}",
            'valid_ri_loss': f"{avg_valid['ri_loss']:.6f}",
            'valid_mag_loss': f"{avg_valid['mag_loss']:.6f}",
            'valid_sisnr': f"{avg_valid['sisnr']:.6f}",
            'valid_pesq': f"{pesq_stoi['pesq']:.4f}",
            'valid_stoi': f"{pesq_stoi['stoi']:.4f}",
            'lr': f'{current_lr:.2e}',
            'scheduler_bad_epochs': scheduler_bad_epochs,
            'time_s': f'{elapsed:.0f}',
        })

        if tb_writer:
            tb_writer.add_scalars('Loss/Total', {'train': avg_train['loss'], 'valid': avg_valid['loss']}, epoch)
            tb_writer.add_scalars('Loss/RI', {'train': avg_train['ri_loss'], 'valid': avg_valid['ri_loss']}, epoch)
            tb_writer.add_scalars('Loss/Mag', {'train': avg_train['mag_loss'], 'valid': avg_valid['mag_loss']}, epoch)
            tb_writer.add_scalars('Loss/SISNR', {'train': avg_train['sisnr'], 'valid': avg_valid['sisnr']}, epoch)
            tb_writer.add_scalar('LR', current_lr, epoch)
            if pesq_stoi['pesq'] > 0:
                tb_writer.add_scalar('Metrics/PESQ', pesq_stoi['pesq'], epoch)
                tb_writer.add_scalar('Metrics/STOI', pesq_stoi['stoi'], epoch)
            tb_writer.flush()

        if wandb is not None and wandb.run is not None:
            wandb_metrics = {
                'epoch': epoch,
                'train/loss': avg_train['loss'],
                'train/ri_loss': avg_train['ri_loss'],
                'train/mag_loss': avg_train['mag_loss'],
                'train/sisnr': avg_train['sisnr'],
                'valid/loss': avg_valid['loss'],
                'valid/ri_loss': avg_valid['ri_loss'],
                'valid/mag_loss': avg_valid['mag_loss'],
                'valid/sisnr': avg_valid['sisnr'],
                'lr': current_lr,
            }
            if pesq_stoi.get('pesq', 0) > 0:
                wandb_metrics['metrics/pesq'] = pesq_stoi['pesq']
                wandb_metrics['metrics/stoi'] = pesq_stoi['stoi']
            wandb.log(wandb_metrics, step=epoch)

        if epoch >= CONFIG.get('early_stopping_min_epochs', 0) and bad_epochs >= CONFIG.get('early_stopping_patience', 15):
            print(f'Early stopping: valid_loss không cải thiện đủ trong {bad_epochs} epoch. best_loss={best_loss:.6f}')
            break

    if tb_writer:
        tb_writer.close()
    if wandb is not None and wandb.run is not None:
        wandb.finish()

    print('\n=== Huấn luyện D3_hybrid_head hoàn tất! ===')
    if best_loss is not None:
        print(f'Best valid loss: {best_loss:.6f}')
    else:
        print('Best valid loss: N/A')
    print(f'Checkpoint: {output_dir}')
    if tb_writer:
        print(f'TensorBoard log: {tb_log_dir}')
else:
    print('run_training=False, bỏ qua training loop.')


## 10. Kiểm tra nhanh sau training

Liệt kê checkpoint/log và nghe thử một mẫu từ validation set.


In [ ]:
import IPython.display as ipd

for name in ['best.pt', 'last.pt', 'config.json', 'notebook_config.json', 'train_log.csv']:
    p = output_dir / name
    if p.exists():
        print(f'{name}: {p} ({p.stat().st_size / 1024:.1f} KB)')
    else:
        print(f'MISSING: {p}')

if (output_dir / 'train_log.csv').exists():
    display(pd.read_csv(output_dir / 'train_log.csv').tail())

best_ckpt_path = output_dir / 'best.pt'
if best_ckpt_path.exists():
    print(f'Tải best checkpoint: {best_ckpt_path}')
    load_checkpoint(best_ckpt_path, model, device=device)

    model.eval()
    with torch.no_grad():
        batch = next(iter(valid_loader))
        noisy = batch['noisy'].to(device)
        clean = batch['clean'].to(device)
        spec = make_stft(noisy[:1], CONFIG['n_fft'], CONFIG['hop_length'], CONFIG['win_length'], window)
        with autocast_context(device, enabled=use_amp):
            pred = model(spec)
        enhanced = make_istft(pred.float(), CONFIG['n_fft'], CONFIG['hop_length'], CONFIG['win_length'], window, length=noisy[:1].shape[-1])

    sr = CONFIG['sample_rate']
    print('Noisy')
    display(ipd.Audio(noisy[0].detach().cpu().numpy(), rate=sr))
    print('Enhanced D3_hybrid_head')
    display(ipd.Audio(enhanced[0].detach().cpu().numpy(), rate=sr))
    print('Clean')
    display(ipd.Audio(clean[0].detach().cpu().numpy(), rate=sr))
else:
    print('Chưa có best.pt để nghe thử.')


## 11. Smoke test và Architecture Benchmark cho D3_hybrid_head

Chỉ chạy `D3_hybrid_head`, không đụng tới baseline/B1/B2/C1 checkpoint.


In [ ]:
import subprocess
import sys


def run_py(args, cwd=Path.cwd(), check=True):
    cmd = [sys.executable, *[str(arg) for arg in args]]
    print('\n$ ' + ' '.join(cmd), flush=True)
    result = subprocess.run(cmd, cwd=str(cwd), text=True)
    if check and result.returncode != 0:
        raise RuntimeError(f'Lệnh lỗi với exit code {result.returncode}: {cmd}')
    return result


AB_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
arch_csv = Path(CONFIG['results_dir']) / 'ablation_arch_benchmark.csv'

if CONFIG.get('run_smoke_tests', True):
    run_py(['ablation/test_ablation_streaming.py', '--configs', CONFIG['config_id']])
else:
    print('run_smoke_tests=False, bỏ qua smoke test.')

if CONFIG.get('run_arch_benchmark', True):
    run_py([
        'ablation/run_ablation_benchmark.py',
        '--output', arch_csv,
        '--configs', CONFIG['config_id'],
        '--device', AB_DEVICE,
        '--frames', '63',
        '--warmup', '1',
        '--repeats', '3',
    ])
else:
    print('run_arch_benchmark=False, bỏ qua architecture benchmark.')

if arch_csv.exists():
    display(pd.read_csv(arch_csv))


## 12. Đánh giá chất lượng D3_hybrid_head: PESQ, STOI, SI-SDR, RTF
Đo lường các chỉ số chất lượng âm thanh sau khi đi qua mô hình:
- **PESQ (Perceptual Evaluation of Speech Quality)**: Đánh giá chất lượng giọng nói cảm nhận (điểm càng cao càng tốt, tối đa 4.5).
- **STOI (Short-Time Objective Intelligibility)**: Đánh giá độ rõ (hiểu được) của giọng nói (điểm từ 0 đến 1, càng cao càng tốt).
- **SI-SDR (Scale-Invariant Signal-to-Distortion Ratio)**: Đánh giá tỷ lệ tín hiệu trên nhiễu (điểm càng cao càng tốt).
- **RTF (Real-Time Factor)**: Tốc độ xử lý (thời gian xử lý / thời gian audio). RTF < 1 nghĩa là xử lý nhanh hơn thời gian thực.

Mục đích của block này là đảm bảo so sánh công bằng với baseline Kaggle V4a: cùng fixed `test.csv`, cùng cách load full-length audio, cùng STFT/ISTFT inference path, cùng metric implementation, cùng cách aggregate mean và cùng file CSV output. Nhờ vậy khác biệt giữa D3_hybrid_head và baseline đến từ mô hình/checkpoint, không đến từ pipeline đánh giá.

Cell này đánh giá trực tiếp `D3_hybrid_head_scratch_v1/best.pt` trên fixed `test.csv`, lưu bảng chi tiết/tổng hợp trong `output_dir/evaluation/`, đồng thời ghi `ablation_quality.csv` để cell tổng hợp kết quả phía sau vẫn chạy được.


In [ ]:
!pip install -q pesq pystoi torchmetrics pandas IPython tqdm

import time
import os
import pandas as pd
import numpy as np
import torch
import torchaudio
import warnings
from pesq import pesq
from pystoi import stoi
from torchmetrics.audio import ScaleInvariantSignalDistortionRatio
from IPython.display import display, FileLink
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

quality_csv = Path(CONFIG['results_dir']) / 'ablation_quality.csv'
best_ckpt = output_dir / 'best.pt'


def metric_mean(df, column):
    values = pd.to_numeric(df[column], errors='coerce').to_numpy(dtype=float)
    finite = np.isfinite(values)
    return float(np.nanmean(values)) if finite.any() else float('nan')


def format_metric(value, digits=4):
    return 'nan' if np.isnan(value) else f'{value:.{digits}f}'


if CONFIG.get('run_quality_eval', True):
    if not best_ckpt.exists():
        raise FileNotFoundError(f'Không tìm thấy checkpoint để eval: {best_ckpt}')

    print(f'Tải trọng số tốt nhất từ: {best_ckpt}')
    load_checkpoint(best_ckpt, model, device=device)
    model.eval()

    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'Model params: {total_params:,} (trainable: {trainable_params:,})')
    print(f'Device: {device}')

    test_csv_path = CONFIG.get('test_csv', str(data_dir / 'test.csv'))
    print(f'Test CSV: {test_csv_path}')
    df_test = pd.read_csv(test_csv_path)

    # Đổi thành số lượng mẫu, ví dụ 50, nếu muốn chạy thử nhanh.
    EVAL_MAX_SAMPLES = None
    if EVAL_MAX_SAMPLES is not None:
        df_test = df_test.head(EVAL_MAX_SAMPLES)
    if df_test.empty:
        raise ValueError(f'Test CSV không có mẫu: {test_csv_path}')
    print(f'Test samples: {len(df_test)}')

    sisdr_metric = ScaleInvariantSignalDistortionRatio().to(device)

    results = []
    total_audio_duration = 0.0
    total_processing_time = 0.0
    EVAL_SR = CONFIG.get('sample_rate', 16000)

    with torch.no_grad():
        for idx, row in tqdm(df_test.iterrows(), total=len(df_test), desc='Evaluating D3_hybrid_head'):
            noisy_wav, sr_n = torchaudio.load(row['noisy_wav'])
            clean_wav, sr_c = torchaudio.load(row['clean_wav'])

            if noisy_wav.shape[0] > 1:
                noisy_wav = noisy_wav.mean(dim=0, keepdim=True)
            if clean_wav.shape[0] > 1:
                clean_wav = clean_wav.mean(dim=0, keepdim=True)

            if sr_n != EVAL_SR:
                noisy_wav = torchaudio.functional.resample(noisy_wav, sr_n, EVAL_SR)
            if sr_c != EVAL_SR:
                clean_wav = torchaudio.functional.resample(clean_wav, sr_c, EVAL_SR)

            noisy_wav = noisy_wav.squeeze(0)
            clean_wav = clean_wav.squeeze(0)
            min_audio_len = min(noisy_wav.numel(), clean_wav.numel())
            if min_audio_len <= 0:
                print(f'Bỏ qua mẫu rỗng: {row.get("ID", idx)}')
                continue
            noisy_wav = noisy_wav[:min_audio_len]
            clean_wav = clean_wav[:min_audio_len]

            audio_duration = noisy_wav.shape[0] / EVAL_SR
            total_audio_duration += audio_duration

            noisy_gpu = noisy_wav.unsqueeze(0).to(device)
            clean_gpu = clean_wav.unsqueeze(0).to(device)

            if device.type == 'cuda':
                torch.cuda.synchronize()
            t_start = time.perf_counter()

            noisy_spec = make_stft(noisy_gpu, CONFIG['n_fft'], CONFIG['hop_length'], CONFIG['win_length'], window)
            amp_enabled = CONFIG.get('use_amp', False) and device.type == 'cuda'
            with autocast_context(device, enabled=amp_enabled):
                pred_stft = model(noisy_spec)

            pred_stft = pred_stft.float()
            enhanced = make_istft(
                pred_stft,
                CONFIG['n_fft'],
                CONFIG['hop_length'],
                CONFIG['win_length'],
                window,
                length=noisy_gpu.shape[-1],
            )

            if device.type == 'cuda':
                torch.cuda.synchronize()
            t_end = time.perf_counter()

            processing_time = t_end - t_start
            total_processing_time += processing_time
            rtf = processing_time / max(audio_duration, 1e-12)

            si_sdr_enhanced = sisdr_metric(enhanced, clean_gpu).item()
            si_sdr_noisy = sisdr_metric(noisy_gpu, clean_gpu).item()

            enhanced_np = enhanced.squeeze(0).detach().cpu().numpy()
            clean_np = clean_wav.numpy()
            noisy_np = noisy_wav.numpy()

            min_len = min(len(enhanced_np), len(clean_np), len(noisy_np))
            enhanced_np = enhanced_np[:min_len]
            clean_np = clean_np[:min_len]
            noisy_np = noisy_np[:min_len]

            try:
                pesq_enhanced = pesq(EVAL_SR, clean_np, enhanced_np, 'wb')
                pesq_noisy = pesq(EVAL_SR, clean_np, noisy_np, 'wb')
            except Exception:
                pesq_enhanced = float('nan')
                pesq_noisy = float('nan')

            try:
                stoi_enhanced = stoi(clean_np, enhanced_np, EVAL_SR, extended=False)
                stoi_noisy = stoi(clean_np, noisy_np, EVAL_SR, extended=False)
            except Exception:
                stoi_enhanced = float('nan')
                stoi_noisy = float('nan')

            results.append({
                'ID': row.get('ID', f'Sample_{idx}'),
                'PESQ_enhanced': round(pesq_enhanced, 4) if not np.isnan(pesq_enhanced) else float('nan'),
                'PESQ_noisy': round(pesq_noisy, 4) if not np.isnan(pesq_noisy) else float('nan'),
                'PESQ_improvement': round(pesq_enhanced - pesq_noisy, 4) if not (np.isnan(pesq_enhanced) or np.isnan(pesq_noisy)) else float('nan'),
                'STOI_enhanced': round(stoi_enhanced, 4) if not np.isnan(stoi_enhanced) else float('nan'),
                'STOI_noisy': round(stoi_noisy, 4) if not np.isnan(stoi_noisy) else float('nan'),
                'STOI_improvement': round(stoi_enhanced - stoi_noisy, 4) if not (np.isnan(stoi_enhanced) or np.isnan(stoi_noisy)) else float('nan'),
                'SI_SDR_enhanced_dB': round(si_sdr_enhanced, 2),
                'SI_SDR_noisy_dB': round(si_sdr_noisy, 2),
                'SI_SDR_improvement_dB': round(si_sdr_enhanced - si_sdr_noisy, 2),
                'RTF': round(rtf, 6),
                'duration_s': round(audio_duration, 3),
            })

    if not results:
        raise RuntimeError('Không đánh giá được mẫu nào trong test set.')

    df_results = pd.DataFrame(results)

    pesq_enh = metric_mean(df_results, 'PESQ_enhanced')
    pesq_noisy = metric_mean(df_results, 'PESQ_noisy')
    pesq_delta = metric_mean(df_results, 'PESQ_improvement')
    stoi_enh = metric_mean(df_results, 'STOI_enhanced')
    stoi_noisy = metric_mean(df_results, 'STOI_noisy')
    stoi_delta = metric_mean(df_results, 'STOI_improvement')
    sisdr_enh = metric_mean(df_results, 'SI_SDR_enhanced_dB')
    sisdr_noisy = metric_mean(df_results, 'SI_SDR_noisy_dB')
    sisdr_delta = metric_mean(df_results, 'SI_SDR_improvement_dB')
    rtf_mean = metric_mean(df_results, 'RTF')
    overall_rtf = total_processing_time / max(total_audio_duration, 1e-12)

    df_summary = pd.DataFrame({
        'Metric': [
            'PESQ (enhanced)', 'PESQ (noisy)', 'PESQ (Δ improvement)',
            'STOI (enhanced)', 'STOI (noisy)', 'STOI (Δ improvement)',
            'SI-SDR enhanced (dB)', 'SI-SDR noisy (dB)', 'SI-SDR Δ (dB)',
            'RTF (mean)', 'RTF (overall)', 'Real-time capable?',
        ],
        'Value': [
            format_metric(pesq_enh, 4),
            format_metric(pesq_noisy, 4),
            format_metric(pesq_delta, 4),
            format_metric(stoi_enh, 4),
            format_metric(stoi_noisy, 4),
            format_metric(stoi_delta, 4),
            format_metric(sisdr_enh, 2),
            format_metric(sisdr_noisy, 2),
            format_metric(sisdr_delta, 2),
            format_metric(rtf_mean, 6),
            format_metric(overall_rtf, 6),
            'Có' if overall_rtf < 1.0 else 'Không',
        ],
    })

    print('\n' + '=' * 60)
    print('  KẾT QUẢ ĐÁNH GIÁ CHẤT LƯỢNG MÔ HÌNH DeepVQE D3_hybrid_head')
    print('=' * 60)
    print(f'Test samples: {len(df_results)}')
    print(f'Model params: {total_params:,}')
    print(f'Device: {device}')
    print(f'Total audio: {total_audio_duration:.1f}s | Processing: {total_processing_time:.1f}s')
    print()
    display(df_summary)

    print('\n--- Chi tiết 10 mẫu đầu tiên ---')
    display(df_results.head(10))

    save_dir = output_dir / 'evaluation'
    save_dir.mkdir(parents=True, exist_ok=True)

    detail_csv = save_dir / 'eval_metrics_per_sample.csv'
    summary_csv_path = save_dir / 'eval_metrics_summary.csv'

    df_results.to_csv(detail_csv, index=False)
    df_summary.to_csv(summary_csv_path, index=False)

    print(f'\nĐã lưu kết quả chi tiết:  {detail_csv}')
    print(f'Đã lưu bảng tổng hợp:     {summary_csv_path}')
    display(FileLink(str(detail_csv)))
    display(FileLink(str(summary_csv_path)))

    quality_csv.parent.mkdir(parents=True, exist_ok=True)
    metadata = reproducibility_metadata(TRAIN_CFG, checkpoint_id=best_ckpt.stem)
    quality_row = {
        'config_id': CONFIG['config_id'],
        **metadata,
        'checkpoint_id': best_ckpt.stem,
        'num_eval_items': len(df_results),
        'eval_duration_s': total_audio_duration,
        'pesq': pesq_enh,
        'stoi': stoi_enh,
        'si_sdr': sisdr_enh,
        'dnsmos_ovrl': '',
        'dnsmos_sig': '',
        'dnsmos_bak': '',
        'erle': '',
        'notes': 'notebook quality eval with PESQ/STOI/SI-SDR/RTF; noisy baselines saved in evaluation detail CSV',
    }
    quality_fields = [
        'config_id',
        'git_commit',
        'config_hash',
        'seed',
        'hardware_info',
        'torch_version',
        'onnxruntime_version',
        'num_threads',
        'checkpoint_id',
        'num_eval_items',
        'eval_duration_s',
        'pesq',
        'stoi',
        'si_sdr',
        'dnsmos_ovrl',
        'dnsmos_sig',
        'dnsmos_bak',
        'erle',
        'notes',
    ]
    if quality_csv.exists():
        existing_quality = pd.read_csv(quality_csv, dtype=str).to_dict('records')
    else:
        existing_quality = []
    existing_quality = [row for row in existing_quality if row.get('config_id') != CONFIG['config_id']]
    existing_quality.append({field: quality_row.get(field, '') for field in quality_fields})
    pd.DataFrame(existing_quality, columns=quality_fields).to_csv(quality_csv, index=False)
    print(f'Đã cập nhật ablation quality CSV: {quality_csv}')
    display(pd.read_csv(quality_csv))
else:
    print('run_quality_eval=False, bỏ qua eval.')
    if quality_csv.exists():
        display(pd.read_csv(quality_csv))


## 13. ONNX export/parity cho D3_hybrid_head (tuỳ chọn)


In [ ]:
onnx_csv = Path(CONFIG['results_dir']) / 'ablation_onnx.csv'
onnx_dir = Path(CONFIG['onnx_dir'])

if CONFIG.get('run_onnx_export', False):
    if not best_ckpt.exists():
        raise FileNotFoundError(f'Không tìm thấy checkpoint để export ONNX: {best_ckpt}')
    run_py([
        'ablation/export_ablation_onnx.py',
        '--config-id', CONFIG['config_id'],
        '--checkpoint', best_ckpt,
        '--output-dir', onnx_dir,
        '--results', onnx_csv,
        '--device', 'cpu',
    ])
else:
    print('run_onnx_export=False, bỏ qua ONNX export.')

if onnx_csv.exists():
    display(pd.read_csv(onnx_csv))


## 14. Tổng hợp CSV và nén kết quả

Tạo `ablation_summary.csv` cho riêng D3_hybrid_head và zip toàn bộ workspace output để tải từ Kaggle.
